<a href="https://colab.research.google.com/github/MayerT1/PIPECAST/blob/flood_mapping/flood_mapping/notebooks/Alaska_PIPECAST_perturbations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction

In this module, we will take a sample image from ICEYE and run several different perturbations of the traditional HYDRAFloods workflows

1. Terrain Correction
  - We will use the USGS 3DEP Digital Elevation Model and will run the terrain correction at the native scale of this map, which is 10 meters
  
2. Speckle Filter

  We will use the p_median filter, and will test windows of different sizes: **window of 3, 5, 7, 13, and 19 pixels**

3. Edge Otsu Algorithm

  - connected_pixel parameter. We will test the following values for this parametetr: **50, 100, 200, 300, 350**
  - initial_threshold parameter. This is determined using a sampling method. We will test the following parameters: **1e2, 1e3, 1e4, 1e5, 1e6**
  - thresh_no_data parameter. This will be determined based on the histogram. We will use **mean - 2 standard deviations, mean - 1 standard deviations, mean + 1 standard deviations, mean + 2 standard deviations**
  - We will use the native scale of the image to run the Edge Otsu algorithm


  After we run all combinations of the 5 values for each four parameters, we will pick the 3 best combinations, then we will use those to test the Canny Edge Detector portion of the Edge Otsu algorithm.

  - canny_threshold
  - canny_sigma
  - canny_lt

# Part 0: Housekeeping and obtaining ICEYE image

In [ ]:
my_gee_folder = 'users/mickymags/watchduty/'

In [ ]:
!pip install ipyleaflet==0.18.2 geemap hydrafloods     # Install hydrafloods and its relevant dependencies
!pip install geemap

In [ ]:
from hydrafloods import corrections
import hydrafloods as hf
import ee
import geemap
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
ee.Authenticate()
ee.Initialize(project = 'servir-sco-assets')

In [ ]:
aoi = ee.FeatureCollection('users/mickymags/watchduty/ak_aoi1')

aoi_geom = aoi.geometry()

# Get the coordinates of the center of the AOI for mapping purposes
aoi_centroid = aoi.geometry().centroid()             # Get the center of the AOI
lon = aoi_centroid.coordinates().get(0).getInfo()    # Extract the longitude from the centroid
lat = aoi_centroid.coordinates().get(1).getInfo()    # Extract the latitude from the centroid

In [ ]:
iceye = hf.Dataset(
    region = aoi_geom,               #my_geom
    start_time = '2025-10-13',
    end_time = '2025-10-15',
    asset_id = 'users/mickymags/watchduty/alaska_iceye_v2'
)

In [ ]:
iceye_first = iceye.collection.first()

In [ ]:
#iceye_med = iceye.collection.median()

In [ ]:
#iceye_med = iceye_med.rename('VV', 'angle').clip(aoi_geom)
iceye_first = iceye_first.rename('VV', 'angle').clip(aoi_geom)

In [ ]:
#iceye_med.bandNames().getInfo()
iceye_first.bandNames().getInfo()

In [ ]:
iceye_vp = {
    'bands': 'VV', #'b1',
    'min': 0,
    'max': 2000
}

# Step 1: Pre-Processing

In [ ]:
from hydrafloods import corrections

### Step 1a: Terrain Correction

In [ ]:
elv = ee.ImageCollection("USGS/3DEP/10m_collection").filterBounds(aoi_geom).mosaic().clip(aoi_geom)
#elv_filt = ee.ImageCollection("USGS/3DEP/10m_collection").filterBounds(aoi_geom)

#elv = elv_filt.mosaic().clip(aoi_geom)

#elv_scale = elv_filt.first().projection().nominalScale().getInfo()


In [ ]:
elv.projection().nominalScale().getInfo()

In [ ]:
#elv_proj.projection().getInfo()

In [ ]:
#print(elv_scale)

In [ ]:
#elv_scale = ee.Number(1000)

In [ ]:
elv_proj = elv.reproject(crs = iceye_first.projection(), scale = 10)

In [ ]:
ic_terrain = corrections.slope_correction(iceye_first, elevation=elv_proj, scale = 10) #elevation=elv_proj

In [ ]:
elv_params = {
    'min': 0,
    'max': 3
}

In [ ]:
Map = geemap.Map(center = (lat, lon), zoom = 11)
Map.addLayer(aoi_geom, {}, 'Area of Interest')
Map.addLayer(iceye_first, iceye_vp, 'ICEYE Reg')
Map.addLayer(ic_terrain, iceye_vp, 'Terrain corrected')
Map.addLayer(elv_proj, elv_params, 'Elevation')

Map.addLayerControl()
Map

### Step 1b: Speckle Filter

In [ ]:
ic_pmedian_3 = hf.p_median(ic_terrain, window=3)
ic_pmedian_5 = hf.p_median(ic_terrain, window=5)
ic_pmedian_7 = hf.p_median(ic_terrain, window=7)
ic_pmedian_9 = hf.p_median(ic_terrain, window=9)
ic_pmedian_11 = hf.p_median(ic_terrain, window=11)

In [ ]:
Map = geemap.Map(center = (lat, lon), zoom = 11)
Map.addLayer(aoi_geom, {}, 'Area of Interest')
Map.addLayer(iceye_first, iceye_vp, 'ICEYE Reg')
Map.addLayer(ic_pmedian_3, iceye_vp, 'ICEYE3')
Map.addLayer(ic_pmedian_5, iceye_vp, 'ICEYE5')
Map.addLayer(ic_pmedian_7, iceye_vp, 'ICEYE7')
Map.addLayer(ic_pmedian_9, iceye_vp, 'ICEYE9')
Map.addLayer(ic_pmedian_11, iceye_vp, 'ICEYE11')

Map.addLayerControl()
Map

In [ ]:
# Export Window 3 t,
geemap.ee_export_image_to_drive(image = ic_pmedian_3,
                                folder = 'WatchDuty',
                                description = 'iceye_kipnuk_pmedian_3',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

# Export Window 5
geemap.ee_export_image_to_drive(image = ic_pmedian_5,
                                folder = 'WatchDuty',
                                description = 'iceye_kipnuk_pmedian_5',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

# Export Window 7
geemap.ee_export_image_to_drive(image = ic_pmedian_7,
                                folder = 'WatchDuty',
                                description = 'iceye_kipnuk_pmedian_7',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

# Export Window 9
geemap.ee_export_image_to_drive(image = ic_pmedian_9,
                                folder = 'WatchDuty',
                                description = 'iceye_kipnuk_pmedian_9',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

# Export Window 11
geemap.ee_export_image_to_drive(image = ic_pmedian_11,
                                folder = 'WatchDuty',
                                description = 'iceye_kipnuk_pmedian_11',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

# Step 2: Determine Initial threshold and Thresh No Data

### Step 2a: Determining the Initial Threshold

In [ ]:
iceye_scale = iceye_first.projection().nominalScale().getInfo()

In [ ]:
iceye_scale

In [ ]:
def init_thresh_determiner(image, roi, numpix, my_scale):
  image_sampled = image.sample(roi, numPixels=numpix, scale=my_scale).toList(numpix).getInfo()

  #features = image_sampled['features']
  vals = []
  for j in range(int(numpix)):#(len(features)):
    #print(j)
    if j < numpix - 0.02*numpix:                    # Flag to stop a weird bug from occuring where earthengine doesn't count all the way to numpix
      feat_of_int = image_sampled[j]
      value = feat_of_int['properties']['VV']
      vals.append(value)

  np_vals = np.array(vals)
  my_mean = np_vals.mean()
  return my_mean

In [ ]:
# Terrain corrected ICEYE image speckle filtered with a window of 3, sampled for 100 pixels
ic_pmedian_3_1e2_threshInit = init_thresh_determiner(ic_pmedian_3, aoi, 1e2, iceye_scale)
ic_pmedian_3_1e3_threshInit = init_thresh_determiner(ic_pmedian_3, aoi, 1e3, iceye_scale)
ic_pmedian_3_1e4_threshInit = init_thresh_determiner(ic_pmedian_3, aoi, 1e4, iceye_scale)
ic_pmedian_3_1e5_threshInit = init_thresh_determiner(ic_pmedian_3, aoi, 1e5, iceye_scale)
ic_pmedian_3_1e6_threshInit = init_thresh_determiner(ic_pmedian_3, aoi, 1e6, iceye_scale)

# Terrain corrected ICEYE image speckle filtered with a window of 5, sampled for 100 pixels
ic_pmedian_5_1e2_threshInit = init_thresh_determiner(ic_pmedian_5, aoi, 1e2, iceye_scale)
ic_pmedian_5_1e3_threshInit = init_thresh_determiner(ic_pmedian_5, aoi, 1e3, iceye_scale)
ic_pmedian_5_1e4_threshInit = init_thresh_determiner(ic_pmedian_5, aoi, 1e4, iceye_scale)
ic_pmedian_5_1e5_threshInit = init_thresh_determiner(ic_pmedian_5, aoi, 1e5, iceye_scale)
ic_pmedian_5_1e6_threshInit = init_thresh_determiner(ic_pmedian_5, aoi, 1e6, iceye_scale)

# Terrain corrected ICEYE image speckle filtered with a window of 7, sampled for 100 pixels
ic_pmedian_7_1e2_threshInit = init_thresh_determiner(ic_pmedian_7, aoi, 1e2, iceye_scale)
ic_pmedian_7_1e3_threshInit = init_thresh_determiner(ic_pmedian_7, aoi, 1e3, iceye_scale)
ic_pmedian_7_1e4_threshInit = init_thresh_determiner(ic_pmedian_7, aoi, 1e4, iceye_scale)
ic_pmedian_7_1e5_threshInit = init_thresh_determiner(ic_pmedian_7, aoi, 1e5, iceye_scale)
ic_pmedian_7_1e6_threshInit = init_thresh_determiner(ic_pmedian_7, aoi, 1e6, iceye_scale)

# Terrain corrected ICEYE image speckle filtered with a window of 9, sampled for 100 pixels
ic_pmedian_9_1e2_threshInit = init_thresh_determiner(ic_pmedian_9, aoi, 1e2, iceye_scale)
ic_pmedian_9_1e3_threshInit = init_thresh_determiner(ic_pmedian_9, aoi, 1e3, iceye_scale)
ic_pmedian_9_1e4_threshInit = init_thresh_determiner(ic_pmedian_9, aoi, 1e4, iceye_scale)
ic_pmedian_9_1e5_threshInit = init_thresh_determiner(ic_pmedian_9, aoi, 1e5, iceye_scale)
ic_pmedian_9_1e6_threshInit = init_thresh_determiner(ic_pmedian_9, aoi, 1e6, iceye_scale)

# Terrain corrected ICEYE image speckle filtered with a window of 11, sampled for 100 pixels
ic_pmedian_11_1e2_threshInit = init_thresh_determiner(ic_pmedian_11, aoi, 1e2, iceye_scale)
ic_pmedian_11_1e3_threshInit = init_thresh_determiner(ic_pmedian_11, aoi, 1e3, iceye_scale)
ic_pmedian_11_1e4_threshInit = init_thresh_determiner(ic_pmedian_11, aoi, 1e4, iceye_scale)
ic_pmedian_11_1e5_threshInit = init_thresh_determiner(ic_pmedian_11, aoi, 1e5, iceye_scale)
ic_pmedian_11_1e6_threshInit = init_thresh_determiner(ic_pmedian_11, aoi, 1e6, iceye_scale)

print('Pmedian window of 3 pixels')
print('100 samples:',ic_pmedian_3_1e2_threshInit)
print('1,000 samples:',ic_pmedian_3_1e3_threshInit)
print('10,000 samples:',ic_pmedian_3_1e4_threshInit)
print('100,000 samples:',ic_pmedian_3_1e5_threshInit)
print('1,000,000 samples:',ic_pmedian_3_1e6_threshInit)

print('--------------------------------------------------')
print('Pmedian window of 5 pixels')
print('100 samples:', ic_pmedian_5_1e2_threshInit)
print('1,000 samples:', ic_pmedian_5_1e3_threshInit)
print('10,000 samples:', ic_pmedian_5_1e4_threshInit)
print('100,000 samples:', ic_pmedian_5_1e5_threshInit)
print('1,000,000 samples:', ic_pmedian_5_1e6_threshInit)

print('--------------------------------------------------')
print('Pmedian window of 7 pixels')
print('100 samples:', ic_pmedian_7_1e2_threshInit)
print('1,000 samples:', ic_pmedian_7_1e3_threshInit)
print('10,000 samples:', ic_pmedian_7_1e4_threshInit)
print('100,000 samples:', ic_pmedian_7_1e5_threshInit)
print('1,000,000 samples:', ic_pmedian_7_1e6_threshInit)

print('--------------------------------------------------')
print('Pmedian window of 9 pixels')
print('100 samples:', ic_pmedian_9_1e2_threshInit)
print('1,000 samples:', ic_pmedian_9_1e3_threshInit)
print('10,000 samples:', ic_pmedian_9_1e4_threshInit)
print('100,000 samples:', ic_pmedian_9_1e5_threshInit)
print('1,000,000 samples:', ic_pmedian_9_1e6_threshInit)

print('----------------------------------------------------')
print('Pmedian window of 11 pixels')
print('100 samples:', ic_pmedian_11_1e2_threshInit)
print('1,000 samples:', ic_pmedian_11_1e3_threshInit)
print('10,000 samples:', ic_pmedian_11_1e4_threshInit)
print('100,000 samples:', ic_pmedian_11_1e5_threshInit)
print('1,000,000 samples:', ic_pmedian_11_1e6_threshInit)

### Visualize Initial Thresholds on Histograms

In [ ]:
#plot_histograms(ic_pmedian_3, ['VV'], 'Pmedian3', aoi_geom, 0, 1000, scale = iceye_scale, n_bins = 1024)
#plt.axvline()

In [ ]:
#plot_histograms(ic_pmedian_5, ['VV'], 'Pmedian5', aoi_geom, 0, 1000, scale = iceye_scale, n_bins = 1024)

In [ ]:
# Create a histogram for each band
def plot_histograms_w_thresholds(image, bands,  aoi_descriptor, region, xmin, xmax, threshold_list, final_thresh, scale=3.46325, n_bins=4096):
    """Plot histograms for each band of an ee.Image."""
    fig, axes = plt.subplots(len(bands), 1, figsize=(8, 4*len(bands)))

    if len(bands) == 1:
        axes = [axes]

    for i, band in enumerate(bands):
        # Reduce region to histogram
        hist_dict = image.select(band).reduceRegion(
            reducer=ee.Reducer.histogram(maxBuckets=n_bins), #maxBuckets=n_bins),
            geometry=region,
            scale=scale,

            bestEffort=True
        ).get(band).getInfo()

        if hist_dict is not None:
            counts = hist_dict['histogram']
            bucketMeans = hist_dict['bucketMeans']

            axes[i].bar(bucketMeans, counts, width=(bucketMeans[1]-bucketMeans[0]))
            axes[i].set_title(f"Histogram of {band} for {aoi_descriptor}")
            axes[i].set_xlabel("Pixel values")
            axes[i].set_ylabel("Frequency")
            axes[i].set_xlim(xmin, xmax) #0, 250
        else:
            axes[i].set_title(f"{band}: No data")

    for j in threshold_list:
      thresh_value = j[0]
      thresh_label = j[1]
      thresh_color = j[2]

      #print(thresh_value)
      #print(thresh_label)
      #print(thresh_color)
      plt.axvline(thresh_value, color=thresh_color, label=thresh_label)

    plt.axvline(final_thresh, color='#000000', label='Final Threshold')


    plt.legend(loc='upper right')
    plt.grid()
    plt.tight_layout()
    plt.show()

**Histogram Visualization for a Window of 3 pixels**

In [ ]:
win3_dict = [[ic_pmedian_3_1e2_threshInit, '100 samples', '#FF0000'],
            [ic_pmedian_3_1e3_threshInit, '1,000 samples', '#00FF00'],
            [ic_pmedian_3_1e4_threshInit, '10,000 samples', '#FFa500'],
            [ic_pmedian_3_1e5_threshInit, '100,000 samples', '#FFFF00'],
             [ic_pmedian_3_1e6_threshInit, '1,000,000 samples', '#800080']]

In [ ]:
plot_histograms_w_thresholds(ic_pmedian_3, ['VV'], 'Pmedian Filter, Window of 3 pixels', aoi_geom, 0, 1200, win3_dict, 614, scale = iceye_scale, n_bins = 1024)

Let's take a closer look at this

In [ ]:
plot_histograms_w_thresholds(ic_pmedian_3, ['VV'], 'Pmedian Filter, Window of 3 pixels', aoi_geom, 560, 600, win3_dict, 614, scale = iceye_scale, n_bins = 1024)

**Histogram Visualization for a Window of 5 pixels**

In [ ]:
win5_dict = [[ic_pmedian_5_1e2_threshInit, '100 samples', '#FF0000'],
            [ic_pmedian_5_1e3_threshInit, '1,000 samples', '#00FF00'],
            [ic_pmedian_5_1e4_threshInit, '10,000 samples', '#FFa500'],
            [ic_pmedian_5_1e5_threshInit, '100,000 samples', '#FFFF00'],
             [ic_pmedian_5_1e6_threshInit, '1,000,000 samples', '#800080']]

In [ ]:
plot_histograms_w_thresholds(ic_pmedian_5, ['VV'], 'Pmedian Filter, Window of 5 pixels', aoi_geom, 0, 1200, win5_dict, 582, scale = iceye_scale, n_bins = 1024)

**Histogram Visualization for a Window of 7 pixels**

In [ ]:
win7_dict = [[ic_pmedian_7_1e2_threshInit, '100 samples', '#FF0000'],
            [ic_pmedian_7_1e3_threshInit, '1,000 samples', '#00FF00'],
            [ic_pmedian_7_1e4_threshInit, '10,000 samples', '#FFa500'],
            [ic_pmedian_7_1e5_threshInit, '100,000 samples', '#FFFF00'],
             [ic_pmedian_7_1e6_threshInit, '1,000,000 samples', '#800080']]

In [ ]:
plot_histograms_w_thresholds(ic_pmedian_7, ['VV'], 'Pmedian Filter, Window of 7 pixels', aoi_geom, 0, 1200, win7_dict, 578, scale = iceye_scale, n_bins = 1024)

**Histogram Visualization for a Window of 9 pixels**

In [ ]:
win9_dict = [[ic_pmedian_9_1e2_threshInit, '100 samples', '#FF0000'],
            [ic_pmedian_9_1e3_threshInit, '1,000 samples', '#00FF00'],
            [ic_pmedian_9_1e4_threshInit, '10,000 samples', '#FFa500'],
            [ic_pmedian_9_1e5_threshInit, '100,000 samples', '#FFFF00'],
             [ic_pmedian_9_1e6_threshInit, '1,000,000 samples', '#800080']]

In [ ]:
plot_histograms_w_thresholds(ic_pmedian_9, ['VV'], 'Pmedian Filter, Window of 9 pixels', aoi_geom, 0, 1200, win9_dict, 569, scale = iceye_scale, n_bins = 1024)

**Histogram Visualization for a Window of 11 pixels**

In [ ]:
win11_dict = [[ic_pmedian_11_1e2_threshInit, '100 samples', '#FF0000'],
            [ic_pmedian_11_1e3_threshInit, '1,000 samples', '#00FF00'],
            [ic_pmedian_11_1e4_threshInit, '10,000 samples', '#FFa500'],
            [ic_pmedian_11_1e5_threshInit, '100,000 samples', '#FFFF00'],
             [ic_pmedian_11_1e6_threshInit, '1,000,000 samples', '#800080']]

In [ ]:
plot_histograms_w_thresholds(ic_pmedian_11, ['VV'], 'Pmedian Filter, Window of 11 pixels', aoi_geom, 0, 1200, win9_dict, 561, scale = iceye_scale, n_bins = 256)

# Step 2b: Determine thresh_no_data for each speckle window value

For each of the above histograms, we want to find the following values

* the mean
* the value 2 standard deviations below the mean
* the value 1 standard deviations below the mean
* the value 1 standard deviation above the mean
* the value 2 standard deviations above the mean

In [ ]:
def value_obtainer(image, roi, numpix, my_scale):
  image_sampled = image.sample(roi, numPixels=numpix, scale=my_scale).toList(numpix).getInfo()

  #features = image_sampled['features']
  vals = []
  for j in range(int(numpix)):#(len(features)):
    #print(j)
    if j < numpix - 0.02*numpix:                    # Flag to stop a weird bug from occuring where earthengine doesn't count all the way to numpix
      feat_of_int = image_sampled[j]
      value = feat_of_int['properties']['VV']
      vals.append(value)

  np_vals = np.array(vals)
  my_mean = np_vals.mean()
  my_stdev = np_vals.std()

  val1 = my_mean - my_stdev
  val2 = my_mean - (0.5 * my_stdev)
  val3 = my_mean
  val4 = my_mean + (0.5 * my_stdev)
  val5 = my_mean + (my_stdev)
  return [val1, val2, val3, val4, val5]

In [ ]:
win3_values = np.floor(np.array(value_obtainer(ic_pmedian_3, aoi, 1e6, iceye_scale)))
win5_values = np.floor(np.array(value_obtainer(ic_pmedian_5, aoi, 1e6, iceye_scale)))
win7_values = np.floor(np.array(value_obtainer(ic_pmedian_7, aoi, 1e6, iceye_scale)))
win9_values = np.floor(np.array(value_obtainer(ic_pmedian_9, aoi, 1e6, iceye_scale)))
win11_values = np.floor(np.array(value_obtainer(ic_pmedian_11, aoi, 1e6, iceye_scale)))

# Create indexed values for easier passing to the Edge Otsu Algorithm
win3_negsig = win3_values[0]
win3_neghalfsig = win3_values[1]
win3_mu = win3_values[2]
win3_poshalfsig = win3_values[3]
win3_possig = win3_values[4]

win5_negsig = win5_values[0]
win5_neghalfsig = win5_values[1]
win5_mu = win5_values[2]
win5_poshalfsig = win5_values[3]
win5_possig = win5_values[4]

win7_negsig = win7_values[0]
win7_neghalfsig = win7_values[1]
win7_mu = win7_values[2]
win7_poshalfsig = win7_values[3]
win7_possig = win7_values[4]

win9_negsig = win9_values[0]
win9_neghalfsig = win9_values[1]
win9_mu = win9_values[2]
win9_poshalfsig = win9_values[3]
win9_possig = win9_values[4]

win11_negsig = win11_values[0]
win11_neghalfsig = win11_values[1]
win11_mu = win11_values[2]
win11_poshalfsig = win11_values[3]
win11_possig = win11_values[4]

# Double Check that win3 values are the same as indexed values
print(win3_values)
print(win5_values)
print(win7_values)
print(win9_values)
print(win11_values)

print([win3_negsig, win3_neghalfsig, win3_mu, win3_poshalfsig, win3_possig])
print([win5_negsig, win5_neghalfsig, win5_mu, win5_poshalfsig, win5_possig])
print([win7_negsig, win7_neghalfsig, win7_mu, win7_poshalfsig, win7_possig])
print([win9_negsig, win9_neghalfsig, win9_mu, win9_poshalfsig, win9_possig])
print([win11_negsig, win11_neghalfsig, win11_mu, win11_poshalfsig, win11_possig])

In [ ]:
# Create a histogram for each band
def plot_histograms_w_values(image, bands,  aoi_descriptor, region, xmin, xmax, value_list, final_thresh, scale=3.46325, n_bins=4096):
    """Plot histograms for each band of an ee.Image."""
    fig, axes = plt.subplots(len(bands), 1, figsize=(8, 4*len(bands)))

    if len(bands) == 1:
        axes = [axes]

    for i, band in enumerate(bands):
        # Reduce region to histogram
        hist_dict = image.select(band).reduceRegion(
            reducer=ee.Reducer.histogram(maxBuckets=n_bins), #maxBuckets=n_bins),
            geometry=region,
            scale=scale,

            bestEffort=True
        ).get(band).getInfo()

        if hist_dict is not None:
            counts = hist_dict['histogram']
            bucketMeans = hist_dict['bucketMeans']

            axes[i].bar(bucketMeans, counts, width=(bucketMeans[1]-bucketMeans[0]))
            axes[i].set_title(f"Histogram and thresh_no_data_values for {aoi_descriptor}")
            axes[i].set_xlabel("Pixel values")
            axes[i].set_ylabel("Frequency")
            axes[i].set_xlim(xmin, xmax) #0, 250
        else:
            axes[i].set_title(f"{band}: No data")

    color_list = ['#FF0000', '#00FF00', '#FFFF00', '#00FFFF', '#FF00FF']
    label_list = ['$\mu - \sigma$', '$\mu - 0.5*\sigma$', '$\mu$', '$\mu + 0.5 * \sigma$', '$\mu + \sigma$']

    for j in range(len(value_list)):
      neg2sigma = value_list[0]
      negsigma = value_list[1]
      mu = value_list [2]
      sigma = value_list[3]
      sigma2 = value_list[4]

      #print(thresh_value)
      #print(thresh_label)
      #print(thresh_color)
      plt.axvline(value_list[j], color=color_list[j], label=label_list[j])

    plt.axvline(final_thresh, color='#000000', label = 'Final Threshold')


    plt.legend(loc='upper right')
    #plt.grid()
    plt.tight_layout()
    plt.show()

In [ ]:
plot_histograms_w_values(ic_pmedian_3, ['VV'], 'PMedian Filter, Window of 3 pixels', aoi_geom, 0, 1200, win3_values, 614, scale=iceye_scale, n_bins=1024)

In [ ]:
plot_histograms_w_values(ic_pmedian_5, ['VV'], 'PMedian Filter, Window of 5 pixels', aoi_geom, 0, 1200, win5_values, 582, scale=iceye_scale, n_bins=1024)

In [ ]:
plot_histograms_w_values(ic_pmedian_7, ['VV'], 'PMedian Filter, Window of 7 pixels', aoi_geom, 0, 1200, win7_values, 578, scale=iceye_scale, n_bins=1024)

In [ ]:
plot_histograms_w_values(ic_pmedian_9, ['VV'], 'PMedian Filter, Window of 9 pixels', aoi_geom, 0, 1200, win9_values, 569, scale=iceye_scale, n_bins=1024)

In [ ]:
plot_histograms_w_values(ic_pmedian_11, ['VV'], 'PMedian Filter, Window of 11 pixels', aoi_geom, 0, 1200, win11_values, 561, scale=iceye_scale, n_bins=1024)

# Step 4: Run all parameterizations

since changing the number of pixels used to sample to find the initial threshold does not seem to impact the initial threshold, we will not parameterize this variable. This is because without parameterizing this variable, we will still have 125 different permutations to test.

Recall that the naming convention we will use to keep the permutations straight is thresholdingAlg_speckleWindow_pointsForInitialSampling_InitThreshold_threshNoDataPosition_threshNoDataValue_connectedPixels

* Window of 3
  * Initial threshold using 10,000 pixels
    * thresh_no_data using mu - sigma
      * 50 connected pxiels
        * This branch corresponds to an initial threshold of 580 and a thresh_no_data value of 256
        * Thus, the file name will be called edgeotsu_Pmedian3_1e4_580_negsigma_356_50
      * 100 connected pixels
      * 200 connected pixels
      * 300 connected pixels
      * 350 connected pixels
    * thresh_no_data using mu - sigma/2
      * 50 connected pixels
      * 100 connected pixels
      * 200 connected pixels
      * 300 connected pixels
      * 350 connected pixels
    * thresh_no_data using mu
      * 50 connected pixels
      * 100 connected pixels
      * 200 connected pixels
      * 300 connected pixels
      * 350 connected pixels
    * thresh_no_data using mu + sigma/2
      * 50 connected pixels
      * 100 connected pixels
      * 200 connected pixels
      * 300 connected pixels
      * 350 connected pixels
    * thresh_no_data using mu + sigma
      * 50 connected pixels
      * 100 connected pixels
      * 200 connected pixels
      * 300 connected pixels
      * 350 connected pixels
* Window of 5
  * copy above tree from Window of 3
* window of 7
  * copy above tree from window of 3
* window of 9
  * copy tree from window 3
* window of 11
  * copy tree from window 3

### 4a: PMedian, Window of 3

**Window of 3, thresh_no_data mu - sigma**

In [ ]:
print('Initial Threshold: {0:0.0f}'.format(ic_pmedian_3_1e4_threshInit))
print('Thresh No Data: {0:0.0f}'.format(win3_negsig))

In [ ]:
ic_pmedian_3

In [ ]:
# Window of 3, thresh_no_data mu - sigma, 50 connected pixels

edgeotsu_pmedian3_1e4_580_negsigma_356_50 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=50,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    return_threshold = True
)

# Window of 3, thresh_no_data mu - sigma, 100 connected pixels
edgeotsu_pmedian3_1e4_580_negsigma_356_100 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=100,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    return_threshold = True
)

edgeotsu_pmedian3_1e4_580_negsigma_356_200 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    return_threshold = True
)

edgeotsu_pmedian3_1e4_580_negsigma_356_300 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=300,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    return_threshold = True
)

edgeotsu_pmedian3_1e4_580_negsigma_356_350 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=350,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom,
    return_threshold = True
)

In [ ]:
thresh_edgeotsu_pmedian3_1e4_580_negsigma_356_50 = edgeotsu_pmedian3_1e4_580_negsigma_356_50.getInfo()['bands'][0]['data_type']['min']
thresh_edgeotsu_pmedian3_1e4_580_negsigma_356_100 = edgeotsu_pmedian3_1e4_580_negsigma_356_100.getInfo()['bands'][0]['data_type']['min']

print(thresh_edgeotsu_pmedian3_1e4_580_negsigma_356_50)
print(thresh_edgeotsu_pmedian3_1e4_580_negsigma_356_100)

**Window of 3, thresh_no_data mu - sigma/2**

In [ ]:
print('Thresh No Data: {0:0.0f}'.format(win3_neghalfsig))

In [ ]:
# Window of 3, thresh_no_data mu - sigma/2

edgeotsu_pmedian3_1e4_580_neghalfsigma_469_50 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=50,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_neghalfsig,
    region = aoi_geom
)

# Window of 3, thresh_no_data mu - sigma, 100 connected pixels
edgeotsu_pmedian3_1e4_580_neghalfsigma_469_100 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=100,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_neghalfsig,
    region = aoi_geom
)

edgeotsu_pmedian3_1e4_580_neghalfsigma_469_200 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_neghalfsig,
    region = aoi_geom
)

edgeotsu_pmedian3_1e4_580_neghalfsigma_469_300 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=300,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom
)

edgeotsu_pmedian3_1e4_580_neghalfsigma_469_350 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=350,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_negsig,
    region = aoi_geom
)

**Window of 3, thresh_no_data mu**

In [ ]:
print('Thresh No Data: {0:0.0f}'.format(win3_mu))

In [ ]:
# Window of 3, thresh_no_data mu - sigma/2

edgeotsu_pmedian3_1e4_580_mu_582_50 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=50,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_mu,
    region = aoi_geom
)

# Window of 3, thresh_no_data mu - sigma, 100 connected pixels
edgeotsu_pmedian3_1e4_580_mu_582_100 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=100,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_mu,
    region = aoi_geom
)

edgeotsu_pmedian3_1e4_580_mu_582_200 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_mu,
    region = aoi_geom
)

edgeotsu_pmedian3_1e4_580_mu_582_300 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=300,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_mu,
    region = aoi_geom
)

edgeotsu_pmedian3_1e4_580_mu_582_350 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=350,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_mu,
    region = aoi_geom
)

**Window of 3, thresh_no_data mu + sigma/2**

In [ ]:
print('Thresh No Data: {0:0.0f}'.format(win3_poshalfsig))

In [ ]:
# Window of 3, thresh_no_data mu - sigma/2

edgeotsu_pmedian3_1e4_580_poshalfsig_695_50 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=50,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_poshalfsig,
    region = aoi_geom
)

# Window of 3, thresh_no_data mu - sigma, 100 connected pixels
edgeotsu_pmedian3_1e4_580_poshalfsigma_695_100 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=100,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_poshalfsig,
    region = aoi_geom
)

edgeotsu_pmedian3_1e4_580_poshalfsigma_695_200 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_poshalfsig,
    region = aoi_geom
)

edgeotsu_pmedian3_1e4_580_poshalfsigma_695_300 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=300,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_poshalfsig,
    region = aoi_geom
)

edgeotsu_pmedian3_1e4_580_poshalfsigma_695_350 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=350,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_poshalfsig,
    region = aoi_geom
)

**Window of 3, thresh_no_data mu + sigma**

In [ ]:
print('Thresh No Data: {0:0.0f}'.format(win3_possig))

In [ ]:
# Window of 3, thresh_no_data mu - sigma/2

edgeotsu_pmedian3_1e4_580_possigma_809_50 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=50,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_possig,
    region = aoi_geom,
    return_threshold=True
)

# Window of 3, thresh_no_data mu - sigma, 100 connected pixels
edgeotsu_pmedian3_1e4_580_possigma_809_100 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=100,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_possig,
    region = aoi_geom
)

edgeotsu_pmedian3_1e4_580_possigma_809_200 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_possig,
    region = aoi_geom
)

edgeotsu_pmedian3_1e4_580_possigma_809_300 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=300,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_possig,
    region = aoi_geom
)

edgeotsu_pmedian3_1e4_580_possigma_809_350 = hf.edge_otsu(
    ic_pmedian_3,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=350,
    initial_threshold=ic_pmedian_3_1e4_threshInit,
    thresh_no_data = win3_possig,
    region = aoi_geom
)

In [ ]:
thresh_edgeotsu_pmedian3_1e4_580_possigma_809_50 = edgeotsu_pmedian3_1e4_580_negsigma_356_50.getInfo()['bands'][0]['data_type']['min']

print(thresh_edgeotsu_pmedian3_1e4_580_possigma_809_50)

In [ ]:
water_vp = {
    'min': 0,
    'max': 1,
    'palette': ['#000000', '#add8e6']
}

In [ ]:
Map = geemap.Map(center = (lat, lon), zoom = 11)
Map.addLayer(aoi_geom, {}, 'Area of Interest')
Map.addLayer(iceye_first, iceye_vp, 'ICEYE Reg')
Map.addLayer(ic_pmedian_3, iceye_vp, 'ICEYE3')
Map.addLayer(edgeotsu_pmedian3_1e4_580_negsigma_356_50, water_vp, '50cp, negsigma, pmed3')
#Map.addLayer(edgeotsu_pmedian3_1e4_580_negsigma_356_100, water_vp, '100cp, negsigma, pmed3')
Map.addLayer(edgeotsu_pmedian3_1e4_580_negsigma_356_350, water_vp, '350cp, negsigma, pmed3')

Map.addLayerControl()
Map

### 4b: PMedian, Window of 5

In [ ]:
print('Initial Threshold: {0:0.0f}'.format(ic_pmedian_5_1e4_threshInit))
print('Thresh No Data: {0:0.0f}'.format(win5_negsig))

In [ ]:
# Window of 5, thresh_no_data mu - sigma, 50 connected pixels

edgeotsu_pmedian5_1e4_573_negsigma_377_50 = hf.edge_otsu(
    ic_pmedian_5,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=50,
    initial_threshold=ic_pmedian_5_1e4_threshInit,
    thresh_no_data = win5_negsig,
    return_threshold=True,
    region = aoi_geom
)

# Window of 3, thresh_no_data mu - sigma, 100 connected pixels
edgeotsu_pmedian5_1e4_573_negsigma_377_100 = hf.edge_otsu(
    ic_pmedian_5,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=100,
    initial_threshold=ic_pmedian_5_1e4_threshInit,
    thresh_no_data = win5_negsig,
    return_threshold=True,
    region = aoi_geom
)

edgeotsu_pmedian5_1e4_573_negsigma_377_200 = hf.edge_otsu(
    ic_pmedian_5,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_5_1e4_threshInit,
    thresh_no_data = win5_negsig,
    return_threshold=True,
    region = aoi_geom
)

edgeotsu_pmedian5_1e4_573_negsigma_377_300 = hf.edge_otsu(
    ic_pmedian_5,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=300,
    initial_threshold=ic_pmedian_5_1e4_threshInit,
    thresh_no_data = win5_negsig,
    return_threshold=True,
    region = aoi_geom
)

edgeotsu_pmedian5_1e4_573_negsigma_377_350 = hf.edge_otsu(
    ic_pmedian_5,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=350,
    initial_threshold=ic_pmedian_5_1e4_threshInit,
    thresh_no_data = win5_negsig,
    return_threshold=True,
    region = aoi_geom
)

In [ ]:
thresh_edgeotsu_pmedian5_1e4_573_negsigma_377_50 = edgeotsu_pmedian5_1e4_573_negsigma_377_50.getInfo()['bands'][0]['data_type']['min']

print(thresh_edgeotsu_pmedian5_1e4_573_negsigma_377_50)

**Window of 5, thresh_no_data mu - sigma/2**

In [ ]:
print('Thresh No Data: {0:0.0f}'.format(win5_neghalfsig))

In [ ]:
# Window of 5, thresh_no_data mu - sigma/2, 50 connected pixels

edgeotsu_pmedian5_1e4_573_neghalfsigma_476_50 = hf.edge_otsu(
    ic_pmedian_5,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=50,
    initial_threshold=ic_pmedian_5_1e4_threshInit,
    thresh_no_data = win5_neghalfsig,
    return_threshold=True,
    region = aoi_geom
)

# Window of 3, thresh_no_data mu - sigma, 100 connected pixels
edgeotsu_pmedian5_1e4_573_neghalfsigma_476_100 = hf.edge_otsu(
    ic_pmedian_5,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=100,
    initial_threshold=ic_pmedian_5_1e4_threshInit,
    thresh_no_data = win5_neghalfsig,
    region = aoi_geom,
    return_threshold=True,
)

edgeotsu_pmedian5_1e4_573_neghalfsigma_476_200 = hf.edge_otsu(
    ic_pmedian_5,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_5_1e4_threshInit,
    thresh_no_data = win5_neghalfsig,
    region = aoi_geom,
    return_threshold=True,
)

edgeotsu_pmedian5_1e4_573_neghalfsigma_476_300 = hf.edge_otsu(
    ic_pmedian_5,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=300,
    initial_threshold=ic_pmedian_5_1e4_threshInit,
    thresh_no_data = win5_neghalfsig,
    region = aoi_geom,
    return_threshold=True,
)

edgeotsu_pmedian5_1e4_573_neghalfsigma_476_350 = hf.edge_otsu(
    ic_pmedian_5,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=350,
    initial_threshold=ic_pmedian_5_1e4_threshInit,
    thresh_no_data = win5_neghalfsig,
    region = aoi_geom,
    return_threshold=True,
)

**Window of 5, thresh_no_data mu**

In [ ]:
print('Thresh No Data: {0:0.0f}'.format(win5_mu))

In [ ]:
# Window of 5, thresh_no_data mu, 50 connected pixels

edgeotsu_pmedian5_1e4_573_mu_575_50 = hf.edge_otsu(
    ic_pmedian_5,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=50,
    initial_threshold=ic_pmedian_5_1e4_threshInit,
    thresh_no_data = win5_mu,
    region = aoi_geom,
    return_threshold=True,
)

# Window of 3, thresh_no_data mu - sigma, 100 connected pixels
edgeotsu_pmedian5_1e4_573_mu_575_100 = hf.edge_otsu(
    ic_pmedian_5,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=100,
    initial_threshold=ic_pmedian_5_1e4_threshInit,
    thresh_no_data = win5_mu,
    region = aoi_geom,
    return_threshold=True,
)

edgeotsu_pmedian5_1e4_573_mu_575_200 = hf.edge_otsu(
    ic_pmedian_5,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_5_1e4_threshInit,
    thresh_no_data = win5_mu,
    region = aoi_geom,
    return_threshold=True,
)

edgeotsu_pmedian5_1e4_573_mu_575_300 = hf.edge_otsu(
    ic_pmedian_5,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=300,
    initial_threshold=ic_pmedian_5_1e4_threshInit,
    thresh_no_data = win5_mu,
    region = aoi_geom,
    return_threshold=True,
)

edgeotsu_pmedian5_1e4_573_mu_575_350 = hf.edge_otsu(
    ic_pmedian_5,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=350,
    initial_threshold=ic_pmedian_5_1e4_threshInit,
    thresh_no_data = win5_mu,
    region = aoi_geom,
    return_threshold=True,
)

**Window of 5, thresh_no_data mu + sigma/2**

In [ ]:
print('Thresh No Data: {0:0.0f}'.format(win5_poshalfsig))

In [ ]:
# Window of 5, thresh_no_data mu, 50 connected pixels

edgeotsu_pmedian5_1e4_573_poshalfsig_674_50 = hf.edge_otsu(
    ic_pmedian_5,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=50,
    initial_threshold=ic_pmedian_5_1e4_threshInit,
    thresh_no_data = win5_poshalfsig,
    region = aoi_geom,
    return_threshold=True,
)

# Window of 3, thresh_no_data mu - sigma, 100 connected pixels
edgeotsu_pmedian5_1e4_573_poshalfsig_674_100 = hf.edge_otsu(
    ic_pmedian_5,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=100,
    initial_threshold=ic_pmedian_5_1e4_threshInit,
    thresh_no_data = win5_poshalfsig,
    region = aoi_geom,
    return_threshold=True,
)

edgeotsu_pmedian5_1e4_573_poshalfsig_674_200 = hf.edge_otsu(
    ic_pmedian_5,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_5_1e4_threshInit,
    thresh_no_data = win5_poshalfsig,
    region = aoi_geom,
    return_threshold=True,
)

edgeotsu_pmedian5_1e4_573_poshalfsig_674_300 = hf.edge_otsu(
    ic_pmedian_5,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=300,
    initial_threshold=ic_pmedian_5_1e4_threshInit,
    thresh_no_data = win5_poshalfsig,
    region = aoi_geom,
    return_threshold=True,
)

edgeotsu_pmedian5_1e4_573_poshalfsig_674_350 = hf.edge_otsu(
    ic_pmedian_5,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=350,
    initial_threshold=ic_pmedian_5_1e4_threshInit,
    thresh_no_data = win5_poshalfsig,
    region = aoi_geom,
    return_threshold=True,
)

**Window of 5, thresh_no_data mu + sigma**

In [ ]:
print('Thresh No Data: {0:0.0f}'.format(win5_possig))

In [ ]:
# Window of 5, thresh_no_data mu, 50 connected pixels

edgeotsu_pmedian5_1e4_573_possig_773_50 = hf.edge_otsu(
    ic_pmedian_5,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=50,
    initial_threshold=ic_pmedian_5_1e4_threshInit,
    thresh_no_data = win5_possig,
    region = aoi_geom,
    return_threshold=True,
)

# Window of 3, thresh_no_data mu - sigma, 100 connected pixels
edgeotsu_pmedian5_1e4_573_possig_773_100 = hf.edge_otsu(
    ic_pmedian_5,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=100,
    initial_threshold=ic_pmedian_5_1e4_threshInit,
    thresh_no_data = win5_possig,
    region = aoi_geom,
    return_threshold=True,
)

edgeotsu_pmedian5_1e4_573_possig_773_200 = hf.edge_otsu(
    ic_pmedian_5,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_5_1e4_threshInit,
    thresh_no_data = win5_possig,
    region = aoi_geom,
    return_threshold=True,
)

edgeotsu_pmedian5_1e4_573_possig_773_300 = hf.edge_otsu(
    ic_pmedian_5,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=300,
    initial_threshold=ic_pmedian_5_1e4_threshInit,
    thresh_no_data = win5_possig,
    region = aoi_geom,
    return_threshold=True,
)

edgeotsu_pmedian5_1e4_573_possig_773_350 = hf.edge_otsu(
    ic_pmedian_5,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=350,
    initial_threshold=ic_pmedian_5_1e4_threshInit,
    thresh_no_data = win5_possig,
    region = aoi_geom,
    return_threshold=True,
)

### 4c: Pmedian, Window of 7

**Window of 7, thresh_no_data mu - sigma**

In [ ]:
print('Initial Threshold: {0:0.0f}'.format(ic_pmedian_7_1e4_threshInit))
print('Thresh No Data: {0:0.0f}'.format(win7_negsig))

In [ ]:
# Window of 7, thresh_no_data mu, 50 connected pixels

edgeotsu_pmedian7_1e4_568_negsig_387_50 = hf.edge_otsu(
    ic_pmedian_7,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=50,
    initial_threshold=ic_pmedian_7_1e4_threshInit,
    thresh_no_data = win7_negsig,
    region = aoi_geom,
    return_threshold=True,
)

# Window of 7, thresh_no_data mu - sigma, 100 connected pixels
edgeotsu_pmedian7_1e4_568_negsig_387_100 = hf.edge_otsu(
    ic_pmedian_7,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=100,
    initial_threshold=ic_pmedian_7_1e4_threshInit,
    thresh_no_data = win7_negsig,
    region = aoi_geom,
    return_threshold=True,
)

edgeotsu_pmedian7_1e4_568_negsig_387_200 = hf.edge_otsu(
    ic_pmedian_7,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_7_1e4_threshInit,
    thresh_no_data = win7_negsig,
    region = aoi_geom,
    return_threshold=True,
)

edgeotsu_pmedian7_1e4_568_negsig_387_300 = hf.edge_otsu(
    ic_pmedian_7,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=300,
    initial_threshold=ic_pmedian_7_1e4_threshInit,
    thresh_no_data = win7_negsig,
    region = aoi_geom,
    return_threshold=True,
)

edgeotsu_pmedian7_1e4_568_negsig_387_350 = hf.edge_otsu(
    ic_pmedian_7,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=350,
    initial_threshold=ic_pmedian_7_1e4_threshInit,
    thresh_no_data = win7_negsig,
    region = aoi_geom,
    return_threshold=True,
)

In [ ]:
thresh_edgeotsu_pmedian7_1e4_568_negsig_387_50 = edgeotsu_pmedian7_1e4_568_negsig_387_50.getInfo()['bands'][0]['data_type']['min']
print(thresh_edgeotsu_pmedian7_1e4_568_negsig_387_50)

**Window of 7, thresh_no_data mu - sigma/2**

In [ ]:
print('Thresh No Data: {0:0.0f}'.format(win7_neghalfsig))

In [ ]:
# Window of 7, thresh_no_data mu, 50 connected pixels

edgeotsu_pmedian7_1e4_568_neghalfsig_479_50 = hf.edge_otsu(
    ic_pmedian_7,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=50,
    initial_threshold=ic_pmedian_7_1e4_threshInit,
    thresh_no_data = win7_neghalfsig,
    region = aoi_geom,
    return_threshold=True,
)

# Window of 7, thresh_no_data mu - sigma, 100 connected pixels
edgeotsu_pmedian7_1e4_568_neghalfsig_479_100 = hf.edge_otsu(
    ic_pmedian_7,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=100,
    initial_threshold=ic_pmedian_7_1e4_threshInit,
    thresh_no_data = win7_neghalfsig,
    region = aoi_geom,
    return_threshold=True,
)

edgeotsu_pmedian7_1e4_568_neghalfsig_479_200 = hf.edge_otsu(
    ic_pmedian_7,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_7_1e4_threshInit,
    thresh_no_data = win7_neghalfsig,
    region = aoi_geom,
    return_threshold=True,
)

edgeotsu_pmedian7_1e4_568_neghalfsig_479_300 = hf.edge_otsu(
    ic_pmedian_7,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=300,
    initial_threshold=ic_pmedian_7_1e4_threshInit,
    thresh_no_data = win7_neghalfsig,
    region = aoi_geom,
    return_threshold=True,
)

edgeotsu_pmedian7_1e4_568_neghalfsig_479_350 = hf.edge_otsu(
    ic_pmedian_7,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=350,
    initial_threshold=ic_pmedian_7_1e4_threshInit,
    thresh_no_data = win7_neghalfsig,
    region = aoi_geom,
    return_threshold=True,
)

**Window of 7, thresh_no_data mu**

In [ ]:
print('Thresh No Data: {0:0.0f}'.format(win7_mu))

In [ ]:
# Window of 7, thresh_no_data mu, 50 connected pixels

edgeotsu_pmedian7_1e4_568_mu_571_50 = hf.edge_otsu(
    ic_pmedian_7,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=50,
    initial_threshold=ic_pmedian_7_1e4_threshInit,
    thresh_no_data = win7_mu,
    region = aoi_geom,
    return_threshold=True,
)

# Window of 7, thresh_no_data mu - sigma, 100 connected pixels
edgeotsu_pmedian7_1e4_568_mu_571_100 = hf.edge_otsu(
    ic_pmedian_7,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=100,
    initial_threshold=ic_pmedian_7_1e4_threshInit,
    thresh_no_data = win7_mu,
    region = aoi_geom,
    return_threshold=True,
)

edgeotsu_pmedian7_1e4_568_mu_571_200 = hf.edge_otsu(
    ic_pmedian_7,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_7_1e4_threshInit,
    thresh_no_data = win7_mu,
    region = aoi_geom,
    return_threshold=True,
)

edgeotsu_pmedian7_1e4_568_mu_571_300 = hf.edge_otsu(
    ic_pmedian_7,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=300,
    initial_threshold=ic_pmedian_7_1e4_threshInit,
    thresh_no_data = win7_mu,
    region = aoi_geom,
    return_threshold=True,
)

edgeotsu_pmedian7_1e4_568_mu_571_350 = hf.edge_otsu(
    ic_pmedian_7,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=350,
    initial_threshold=ic_pmedian_7_1e4_threshInit,
    thresh_no_data = win7_mu,
    region = aoi_geom,
    return_threshold=True,
)

**Window of 7, thresh_no_data mu + sigma/2**

In [ ]:
print('Thresh No Data: {0:0.0f}'.format(win7_poshalfsig))

In [ ]:
# Window of 7, thresh_no_data mu, 50 connected pixels

edgeotsu_pmedian7_1e4_568_poshalfsig_663_50 = hf.edge_otsu(
    ic_pmedian_7,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=50,
    initial_threshold=ic_pmedian_7_1e4_threshInit,
    thresh_no_data = win7_poshalfsig,
    region = aoi_geom,
    return_threshold=True,
)

# Window of 7, thresh_no_data mu - sigma, 100 connected pixels
edgeotsu_pmedian7_1e4_568_poshalfsig_663_100 = hf.edge_otsu(
    ic_pmedian_7,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=100,
    initial_threshold=ic_pmedian_7_1e4_threshInit,
    thresh_no_data = win7_poshalfsig,
    region = aoi_geom,
    return_threshold=True,
)

edgeotsu_pmedian7_1e4_568_poshalfsig_663_200 = hf.edge_otsu(
    ic_pmedian_7,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_7_1e4_threshInit,
    thresh_no_data = win7_poshalfsig,
    region = aoi_geom,
    return_threshold=True,
)

edgeotsu_pmedian7_1e4_568_poshalfsig_663_300 = hf.edge_otsu(
    ic_pmedian_7,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=300,
    initial_threshold=ic_pmedian_7_1e4_threshInit,
    thresh_no_data = win7_poshalfsig,
    region = aoi_geom,
    return_threshold=True,
)

edgeotsu_pmedian7_1e4_568_poshalfsig_663_350 = hf.edge_otsu(
    ic_pmedian_7,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=350,
    initial_threshold=ic_pmedian_7_1e4_threshInit,
    thresh_no_data = win7_poshalfsig,
    region = aoi_geom,
    return_threshold=True,
)

**Window of 7, thresh_no_data mu + sigma**

In [ ]:
print('Thresh No Data: {0:0.0f}'.format(win7_possig))

In [ ]:
# Window of 7, thresh_no_data mu, 50 connected pixels

edgeotsu_pmedian7_1e4_568_possig_754_50 = hf.edge_otsu(
    ic_pmedian_7,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=50,
    initial_threshold=ic_pmedian_7_1e4_threshInit,
    thresh_no_data = win7_possig,
    region = aoi_geom,
    return_threshold=True,
)

# Window of 7, thresh_no_data mu - sigma, 100 connected pixels
edgeotsu_pmedian7_1e4_568_possig_754_100 = hf.edge_otsu(
    ic_pmedian_7,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=100,
    initial_threshold=ic_pmedian_7_1e4_threshInit,
    thresh_no_data = win7_possig,
    region = aoi_geom,
    return_threshold=True,
)

edgeotsu_pmedian7_1e4_568_possig_754_200 = hf.edge_otsu(
    ic_pmedian_7,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_7_1e4_threshInit,
    thresh_no_data = win7_possig,
    region = aoi_geom,
    return_threshold=True,
)

edgeotsu_pmedian7_1e4_568_possig_754_300 = hf.edge_otsu(
    ic_pmedian_7,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=300,
    initial_threshold=ic_pmedian_7_1e4_threshInit,
    thresh_no_data = win7_possig,
    region = aoi_geom,
    return_threshold=True,
)

edgeotsu_pmedian7_1e4_568_possig_754_350 = hf.edge_otsu(
    ic_pmedian_7,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=350,
    initial_threshold=ic_pmedian_7_1e4_threshInit,
    thresh_no_data = win7_possig,
    region = aoi_geom,
    return_threshold=True,
)

### 4d: Pmedian, Window of 9

**Window of 9, thresh_no_data mu - sigma**

In [ ]:
print('Initial Threshold: {0:0.0f}'.format(ic_pmedian_9_1e4_threshInit))
print('Thresh No Data: {0:0.0f}'.format(win9_negsig))

In [ ]:
# Window of 9, thresh_no_data mu, 50 connected pixels

edgeotsu_pmedian9_1e4_566_negsig_394_50 = hf.edge_otsu(
    ic_pmedian_9,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=50,
    initial_threshold=ic_pmedian_9_1e4_threshInit,
    thresh_no_data = win9_negsig,
    region = aoi_geom,
    return_threshold=True,
)

# Window of 9, thresh_no_data mu - sigma, 100 connected pixels
edgeotsu_pmedian9_1e4_566_negsig_394_100 = hf.edge_otsu(
    ic_pmedian_9,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=100,
    initial_threshold=ic_pmedian_9_1e4_threshInit,
    thresh_no_data = win9_negsig,
    region = aoi_geom,
    return_threshold=True,
)

edgeotsu_pmedian9_1e4_566_negsig_394_200 = hf.edge_otsu(
    ic_pmedian_9,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_9_1e4_threshInit,
    thresh_no_data = win9_negsig,
    region = aoi_geom
)

edgeotsu_pmedian9_1e4_566_negsig_394_300 = hf.edge_otsu(
    ic_pmedian_9,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=300,
    initial_threshold=ic_pmedian_9_1e4_threshInit,
    thresh_no_data = win9_negsig,
    region = aoi_geom
)

edgeotsu_pmedian9_1e4_566_negsig_394_350 = hf.edge_otsu(
    ic_pmedian_9,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=350,
    initial_threshold=ic_pmedian_9_1e4_threshInit,
    thresh_no_data = win9_negsig,
    region = aoi_geom
)

In [ ]:
thresh_edgeotsu_pmedian9_1e4_566_negsig_394_50 = edgeotsu_pmedian9_1e4_566_negsig_394_50.getInfo()['bands'][0]['data_type']['min']
print(thresh_edgeotsu_pmedian9_1e4_566_negsig_394_50)

**Window of 9, thresh_no_data mu - sigma/2**

In [ ]:
print('Thresh No Data: {0:0.0f}'.format(win9_neghalfsig))

In [ ]:
# Window of 9, thresh_no_data mu, 50 connected pixels

edgeotsu_pmedian9_1e4_566_neghalfsig_481_50 = hf.edge_otsu(
    ic_pmedian_9,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=50,
    initial_threshold=ic_pmedian_9_1e4_threshInit,
    thresh_no_data = win9_neghalfsig,
    region = aoi_geom
)

# Window of 9, thresh_no_data mu - sigma, 100 connected pixels
edgeotsu_pmedian9_1e4_566_neghalfsig_481_100 = hf.edge_otsu(
    ic_pmedian_9,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=100,
    initial_threshold=ic_pmedian_9_1e4_threshInit,
    thresh_no_data = win9_neghalfsig,
    region = aoi_geom
)

edgeotsu_pmedian9_1e4_566_neghalfsig_481_200 = hf.edge_otsu(
    ic_pmedian_9,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_9_1e4_threshInit,
    thresh_no_data = win9_neghalfsig,
    region = aoi_geom
)

edgeotsu_pmedian9_1e4_566_neghalfsig_481_300 = hf.edge_otsu(
    ic_pmedian_9,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=300,
    initial_threshold=ic_pmedian_9_1e4_threshInit,
    thresh_no_data = win9_neghalfsig,
    region = aoi_geom
)

edgeotsu_pmedian9_1e4_566_neghalfsig_481_350 = hf.edge_otsu(
    ic_pmedian_9,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=350,
    initial_threshold=ic_pmedian_9_1e4_threshInit,
    thresh_no_data = win9_neghalfsig,
    region = aoi_geom
)

**Window of 9, thresh_no_data mu**

In [ ]:
print('Thresh No Data: {0:0.0f}'.format(win9_mu))

In [ ]:
# Window of 9, thresh_no_data mu, 50 connected pixels

edgeotsu_pmedian9_1e4_566_mu_568_50 = hf.edge_otsu(
    ic_pmedian_9,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=50,
    initial_threshold=ic_pmedian_9_1e4_threshInit,
    thresh_no_data = win9_mu,
    region = aoi_geom
)

# Window of 9, thresh_no_data mu - sigma, 100 connected pixels
edgeotsu_pmedian9_1e4_566_mu_568_100 = hf.edge_otsu(
    ic_pmedian_9,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=100,
    initial_threshold=ic_pmedian_9_1e4_threshInit,
    thresh_no_data = win9_mu,
    region = aoi_geom
)

edgeotsu_pmedian9_1e4_566_mu_568_200 = hf.edge_otsu(
    ic_pmedian_9,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_9_1e4_threshInit,
    thresh_no_data = win9_mu,
    region = aoi_geom
)

edgeotsu_pmedian9_1e4_566_mu_568_300 = hf.edge_otsu(
    ic_pmedian_9,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=300,
    initial_threshold=ic_pmedian_9_1e4_threshInit,
    thresh_no_data = win9_mu,
    region = aoi_geom
)

edgeotsu_pmedian9_1e4_566_mu_568_350 = hf.edge_otsu(
    ic_pmedian_9,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=350,
    initial_threshold=ic_pmedian_9_1e4_threshInit,
    thresh_no_data = win9_mu,
    region = aoi_geom
)

**Window of 9, thresh_no_data mu + sigma/2**

In [ ]:
print('Thresh No Data: {0:0.0f}'.format(win9_poshalfsig))

In [ ]:
# Window of 9, thresh_no_data mu, 50 connected pixels

edgeotsu_pmedian9_1e4_566_poshalfsig_655_50 = hf.edge_otsu(
    ic_pmedian_9,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=50,
    initial_threshold=ic_pmedian_9_1e4_threshInit,
    thresh_no_data = win9_poshalfsig,
    region = aoi_geom
)

# Window of 9, thresh_no_data mu - sigma, 100 connected pixels
edgeotsu_pmedian9_1e4_566_poshalfsig_655_100 = hf.edge_otsu(
    ic_pmedian_9,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=100,
    initial_threshold=ic_pmedian_9_1e4_threshInit,
    thresh_no_data = win9_poshalfsig,
    region = aoi_geom
)

edgeotsu_pmedian9_1e4_566_poshalfsig_655_200 = hf.edge_otsu(
    ic_pmedian_9,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_9_1e4_threshInit,
    thresh_no_data = win9_poshalfsig,
    region = aoi_geom
)

edgeotsu_pmedian9_1e4_566_poshalfsig_655_300 = hf.edge_otsu(
    ic_pmedian_9,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=300,
    initial_threshold=ic_pmedian_9_1e4_threshInit,
    thresh_no_data = win9_poshalfsig,
    region = aoi_geom
)

edgeotsu_pmedian9_1e4_566_poshalfsig_655_350 = hf.edge_otsu(
    ic_pmedian_9,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=350,
    initial_threshold=ic_pmedian_9_1e4_threshInit,
    thresh_no_data = win9_poshalfsig,
    region = aoi_geom
)

**Window of 9, thresh_no_data mu + sigma**

In [ ]:
print('Thresh No Data: {0:0.0f}'.format(win9_possig))

In [ ]:
# Window of 9, thresh_no_data mu, 50 connected pixels

edgeotsu_pmedian9_1e4_566_possig_742_50 = hf.edge_otsu(
    ic_pmedian_9,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=50,
    initial_threshold=ic_pmedian_9_1e4_threshInit,
    thresh_no_data = win9_possig,
    region = aoi_geom
)

# Window of 9, thresh_no_data mu - sigma, 100 connected pixels
edgeotsu_pmedian9_1e4_566_possig_742_100 = hf.edge_otsu(
    ic_pmedian_9,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=100,
    initial_threshold=ic_pmedian_9_1e4_threshInit,
    thresh_no_data = win9_possig,
    region = aoi_geom
)

edgeotsu_pmedian9_1e4_566_possig_742_200 = hf.edge_otsu(
    ic_pmedian_9,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_9_1e4_threshInit,
    thresh_no_data = win9_possig,
    region = aoi_geom
)

edgeotsu_pmedian9_1e4_566_possig_742_300 = hf.edge_otsu(
    ic_pmedian_9,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=300,
    initial_threshold=ic_pmedian_9_1e4_threshInit,
    thresh_no_data = win9_possig,
    region = aoi_geom
)

edgeotsu_pmedian9_1e4_566_possig_742_350 = hf.edge_otsu(
    ic_pmedian_9,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=350,
    initial_threshold=ic_pmedian_9_1e4_threshInit,
    thresh_no_data = win9_possig,
    region = aoi_geom
)

### 4e: Pmedian, Window of 11

**Window of 11, thresh_no_data mu - sigma**

In [ ]:
print('Initial Threshold: {0:0.0f}'.format(ic_pmedian_11_1e4_threshInit))
print('Thresh No Data: {0:0.0f}'.format(win11_negsig))

In [ ]:
# Window of 11, thresh_no_data mu, 50 connected pixels

edgeotsu_pmedian11_1e4_563_negsig_394_50 = hf.edge_otsu(
    ic_pmedian_11,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=50,
    initial_threshold=ic_pmedian_11_1e4_threshInit,
    thresh_no_data = win11_negsig,
    region = aoi_geom,
    return_threshold=True
)

# Window of 11, thresh_no_data mu - sigma, 100 connected pixels
edgeotsu_pmedian11_1e4_563_negsig_394_100 = hf.edge_otsu(
    ic_pmedian_11,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=100,
    initial_threshold=ic_pmedian_11_1e4_threshInit,
    thresh_no_data = win11_negsig,
    region = aoi_geom
)

edgeotsu_pmedian11_1e4_563_negsig_394_200 = hf.edge_otsu(
    ic_pmedian_11,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_11_1e4_threshInit,
    thresh_no_data = win11_negsig,
    region = aoi_geom
)

edgeotsu_pmedian11_1e4_563_negsig_394_300 = hf.edge_otsu(
    ic_pmedian_11,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=300,
    initial_threshold=ic_pmedian_11_1e4_threshInit,
    thresh_no_data = win11_negsig,
    region = aoi_geom
)

edgeotsu_pmedian11_1e4_563_negsig_394_350 = hf.edge_otsu(
    ic_pmedian_11,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=350,
    initial_threshold=ic_pmedian_11_1e4_threshInit,
    thresh_no_data = win11_negsig,
    region = aoi_geom
)

In [ ]:
thresh_edgeotsu_pmedian11_1e4_563_negsig_394_50 = edgeotsu_pmedian11_1e4_563_negsig_394_50.getInfo()['bands'][0]['data_type']['min']
print(thresh_edgeotsu_pmedian11_1e4_563_negsig_394_50)

**Window of 11, thresh_no_data mu - sigma/2**

In [ ]:
print('Thresh No Data: {0:0.0f}'.format(win11_neghalfsig))

In [ ]:
# Window of 11, thresh_no_data mu, 50 connected pixels

edgeotsu_pmedian11_1e4_563_neghalfsig_482_50 = hf.edge_otsu(
    ic_pmedian_11,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=50,
    initial_threshold=ic_pmedian_11_1e4_threshInit,
    thresh_no_data = win11_neghalfsig,
    region = aoi_geom
)

# Window of 11, thresh_no_data mu - sigma, 100 connected pixels
edgeotsu_pmedian11_1e4_563_neghalfsig_482_100 = hf.edge_otsu(
    ic_pmedian_11,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=100,
    initial_threshold=ic_pmedian_11_1e4_threshInit,
    thresh_no_data = win11_neghalfsig,
    region = aoi_geom
)

edgeotsu_pmedian11_1e4_563_neghalfsig_482_200 = hf.edge_otsu(
    ic_pmedian_11,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_11_1e4_threshInit,
    thresh_no_data = win11_neghalfsig,
    region = aoi_geom
)

edgeotsu_pmedian11_1e4_563_neghalfsig_482_300 = hf.edge_otsu(
    ic_pmedian_11,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=300,
    initial_threshold=ic_pmedian_11_1e4_threshInit,
    thresh_no_data = win11_neghalfsig,
    region = aoi_geom
)

edgeotsu_pmedian11_1e4_563_neghalfsig_482_350 = hf.edge_otsu(
    ic_pmedian_11,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=350,
    initial_threshold=ic_pmedian_11_1e4_threshInit,
    thresh_no_data = win11_neghalfsig,
    region = aoi_geom
)

**Window of 11, thresh_no_data mu**

In [ ]:
print('Thresh No Data: {0:0.0f}'.format(win11_mu))

In [ ]:
# Window of 11, thresh_no_data mu, 50 connected pixels

edgeotsu_pmedian11_1e4_563_mu_566_50 = hf.edge_otsu(
    ic_pmedian_11,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=50,
    initial_threshold=ic_pmedian_11_1e4_threshInit,
    thresh_no_data = win11_mu,
    region = aoi_geom
)

# Window of 11, thresh_no_data mu - sigma, 100 connected pixels
edgeotsu_pmedian11_1e4_563_mu_566_100 = hf.edge_otsu(
    ic_pmedian_11,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=100,
    initial_threshold=ic_pmedian_11_1e4_threshInit,
    thresh_no_data = win11_mu,
    region = aoi_geom
)

edgeotsu_pmedian11_1e4_563_mu_566_200 = hf.edge_otsu(
    ic_pmedian_11,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_11_1e4_threshInit,
    thresh_no_data = win11_mu,
    region = aoi_geom
)

edgeotsu_pmedian11_1e4_563_mu_566_300 = hf.edge_otsu(
    ic_pmedian_11,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=300,
    initial_threshold=ic_pmedian_11_1e4_threshInit,
    thresh_no_data = win11_mu,
    region = aoi_geom
)

edgeotsu_pmedian11_1e4_563_mu_566_350 = hf.edge_otsu(
    ic_pmedian_11,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=350,
    initial_threshold=ic_pmedian_11_1e4_threshInit,
    thresh_no_data = win11_mu,
    region = aoi_geom
)

**Window of 11, thresh_no_data mu + sigma/2**

In [ ]:
print('Thresh No Data: {0:0.0f}'.format(win11_poshalfsig))

In [ ]:
# Window of 11, thresh_no_data mu, 50 connected pixels

edgeotsu_pmedian11_1e4_563_poshalfsig_649_50 = hf.edge_otsu(
    ic_pmedian_11,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=50,
    initial_threshold=ic_pmedian_11_1e4_threshInit,
    thresh_no_data = win11_poshalfsig,
    region = aoi_geom
)

# Window of 11, thresh_no_data mu - sigma, 100 connected pixels
edgeotsu_pmedian11_1e4_563_poshalfsig_649_100 = hf.edge_otsu(
    ic_pmedian_11,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=100,
    initial_threshold=ic_pmedian_11_1e4_threshInit,
    thresh_no_data = win11_poshalfsig,
    region = aoi_geom
)

edgeotsu_pmedian11_1e4_563_poshalfsig_649_200 = hf.edge_otsu(
    ic_pmedian_11,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_11_1e4_threshInit,
    thresh_no_data = win11_poshalfsig,
    region = aoi_geom
)

edgeotsu_pmedian11_1e4_563_poshalfsig_649_300 = hf.edge_otsu(
    ic_pmedian_11,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=300,
    initial_threshold=ic_pmedian_11_1e4_threshInit,
    thresh_no_data = win11_poshalfsig,
    region = aoi_geom
)

edgeotsu_pmedian11_1e4_563_poshalfsig_649_350 = hf.edge_otsu(
    ic_pmedian_11,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=350,
    initial_threshold=ic_pmedian_11_1e4_threshInit,
    thresh_no_data = win11_poshalfsig,
    region = aoi_geom
)

**Window of 11, thresh_no_data mu + sigma**

In [ ]:
print('Thresh No Data: {0:0.0f}'.format(win11_possig))

In [ ]:
# Window of 11, thresh_no_data mu, 50 connected pixels

edgeotsu_pmedian11_1e4_563_possig_732_50 = hf.edge_otsu(
    ic_pmedian_11,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=50,
    initial_threshold=ic_pmedian_11_1e4_threshInit,
    thresh_no_data = win11_possig,
    region = aoi_geom
)

# Window of 11, thresh_no_data mu - sigma, 100 connected pixels
edgeotsu_pmedian11_1e4_563_possig_732_100 = hf.edge_otsu(
    ic_pmedian_11,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=100,
    initial_threshold=ic_pmedian_11_1e4_threshInit,
    thresh_no_data = win11_possig,
    region = aoi_geom
)

edgeotsu_pmedian11_1e4_563_possig_732_200 = hf.edge_otsu(
    ic_pmedian_11,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=200,
    initial_threshold=ic_pmedian_11_1e4_threshInit,
    thresh_no_data = win11_possig,
    region = aoi_geom
)

edgeotsu_pmedian11_1e4_563_possig_732_300 = hf.edge_otsu(
    ic_pmedian_11,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=300,
    initial_threshold=ic_pmedian_11_1e4_threshInit,
    thresh_no_data = win11_possig,
    region = aoi_geom
)

edgeotsu_pmedian11_1e4_563_possig_732_350 = hf.edge_otsu(
    ic_pmedian_11,
    band = 'VV',
    scale = iceye_scale,
    connected_pixels=350,
    initial_threshold=ic_pmedian_11_1e4_threshInit,
    thresh_no_data = win11_possig,
    region = aoi_geom
)

# Step 5: Exporting images to Google Drive

### Step 5a: P-Median Window of 3

In [ ]:
##################### P-Median 3, thresh_no_data mu - sigma ########################

# ConnPix 50
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian3_1e4_580_negsigma_356_50,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian3_1e4_580_negsigma_356_50',
                                description = 'edgeotsu_pmedian3_1e4_580_negsigma_356_50',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 100
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian3_1e4_580_negsigma_356_100,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian3_1e4_580_negsigma_356_100',
                                description = 'edgeotsu_pmedian3_1e4_580_negsigma_356_100',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 200
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian3_1e4_580_negsigma_356_200,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian3_1e4_580_negsigma_356_200',
                                description = 'edgeotsu_pmedian3_1e4_580_negsigma_356_200',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 300
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian3_1e4_580_negsigma_356_300,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian3_1e4_580_negsigma_356_300',
                                description = 'edgeotsu_pmedian3_1e4_580_negsigma_356_300',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 350
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian3_1e4_580_negsigma_356_350,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian3_1e4_580_negsigma_356_350',
                                description = 'edgeotsu_pmedian3_1e4_580_negsigma_356_350',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)



In [ ]:
##################### P-Median 3, thresh_no_data mu - sigma/2 ########################
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian3_1e4_580_neghalfsigma_469_50,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian3_1e4_580_neghalfsigma_469_50',
                                description = 'edgeotsu_pmedian3_1e4_580_neghalfsigma_469_50',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 100
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian3_1e4_580_neghalfsigma_469_100,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian3_1e4_580_neghalfsigma_469_100',
                                description = 'edgeotsu_pmedian3_1e4_580_neghalfsigma_469_100',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 200
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian3_1e4_580_neghalfsigma_469_200,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian3_1e4_580_neghalfsigma_469_200',
                                description = 'edgeotsu_pmedian3_1e4_580_neghalfsigma_469_200',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 300
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian3_1e4_580_neghalfsigma_469_300,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian3_1e4_580_neghalfsigma_469_300',
                                description = 'edgeotsu_pmedian3_1e4_580_neghalfsigma_469_300',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 350
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian3_1e4_580_neghalfsigma_469_350,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian3_1e4_580_neghalfsigma_469_350',
                                description = 'edgeotsu_pmedian3_1e4_580_neghalfsigma_469_350',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

##################### P-Median 3, thresh_no_data mu ########################
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian3_1e4_580_mu_582_50,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian3_1e4_580_mu_582_50',
                                description = 'edgeotsu_pmedian3_1e4_580_mu_582_50',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 100
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian3_1e4_580_mu_582_100,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian3_1e4_580_mu_582_100',
                                description = 'edgeotsu_pmedian3_1e4_580_mu_582_100',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 200
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian3_1e4_580_mu_582_200,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian3_1e4_580_mu_582_200',
                                description = 'edgeotsu_pmedian3_1e4_580_mu_582_200',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 300
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian3_1e4_580_mu_582_300,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian3_1e4_580_mu_582_300',
                                description = 'edgeotsu_pmedian3_1e4_580_mu_582_300',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 350
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian3_1e4_580_mu_582_350,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian3_1e4_580_mu_582_350',
                                description = 'edgeotsu_pmedian3_1e4_580_mu_582_350',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

##################### P-Median 3, thresh_no_data mu + sigma/2 ########################
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian3_1e4_580_poshalfsig_695_50,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian3_1e4_580_poshalfsig_695_50',
                                description = 'edgeotsu_pmedian3_1e4_580_poshalfsig_695_50',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 100
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian3_1e4_580_poshalfsigma_695_100,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian3_1e4_580_poshalfsig_695_100',
                                description = 'edgeotsu_pmedian3_1e4_580_poshalfsig_695_100',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 200
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian3_1e4_580_poshalfsigma_695_200,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian3_1e4_580_poshalfsig_695_200',
                                description = 'edgeotsu_pmedian3_1e4_580_poshalfsigma_695_200',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 300
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian3_1e4_580_poshalfsigma_695_300,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian3_1e4_580_poshalfsig_695_300',
                                description = 'edgeotsu_pmedian3_1e4_580_poshalfsigma_695_300',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 350
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian3_1e4_580_poshalfsigma_695_350,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian3_1e4_580_poshalfsig_695_350',
                                description = 'edgeotsu_pmedian3_1e4_580_poshalfsigma_695_350',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

##################### P-Median 3, thresh_no_data mu + sigma ########################

geemap.ee_export_image_to_asset(image = edgeotsu_pmedian3_1e4_580_possigma_809_50,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian3_1e4_580_possig_809_50',
                                description = 'edgeotsu_pmedian3_1e4_580_possig_809_50',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 100
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian3_1e4_580_possigma_809_100,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian3_1e4_580_possig_809_100',
                                description = 'edgeotsu_pmedian3_1e4_580_possig_809_100',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 200
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian3_1e4_580_possigma_809_200,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian3_1e4_580_possig_809_200',
                                description = 'edgeotsu_pmedian3_1e4_580_possig_809_200',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 300
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian3_1e4_580_possigma_809_300,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian3_1e4_580_possig_809_300',
                                description = 'edgeotsu_pmedian3_1e4_580_possig_809_300',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 350
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian3_1e4_580_possigma_809_350,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian3_1e4_580_possig_809_350',
                                description = 'edgeotsu_pmedian3_1e4_580_possig_809_350',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

### Step 5b: P-Median Window of 5

In [ ]:
##################### P-Median 5, thresh_no_data mu - sigma ########################

geemap.ee_export_image_to_asset(image = edgeotsu_pmedian5_1e4_573_negsigma_377_50,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian5_1e4_573_negsig_377_50',
                                description = 'edgeotsu_pmedian5_1e4_573_negsig_377_50',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 100
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian5_1e4_573_negsigma_377_100,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian5_1e4_573_negsig_377_100',
                                description = 'edgeotsu_pmedian5_1e4_573_negsig_377_100',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 200
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian5_1e4_573_negsigma_377_200,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian5_1e4_573_negsig_377_200',
                                description = 'edgeotsu_pmedian5_1e4_573_negsig_377_200',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 300
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian5_1e4_573_negsigma_377_300,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian5_1e4_573_negsig_377_300',
                                description = 'edgeotsu_pmedian5_1e4_573_negsig_377_300',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 350
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian5_1e4_573_negsigma_377_350,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian5_1e4_573_negsig_377_350',
                                description = 'edgeotsu_pmedian5_1e4_573_negsig_377_350',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

##################### P-Median 5, thresh_no_data mu - sigma/2 ########################

#ConnPix 50
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian5_1e4_573_neghalfsigma_476_50,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian5_1e4_573_neghalfsig_476_50',
                                description = 'edgeotsu_pmedian5_1e4_573_neghalfsig_476_50',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 100
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian5_1e4_573_neghalfsigma_476_100,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian5_1e4_573_neghalfsig_476_100',
                                description = 'edgeotsu_pmedian5_1e4_573_neghalfsig_476_100',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 200
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian5_1e4_573_neghalfsigma_476_200,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian5_1e4_573_neghalfsig_476_200',
                                description = 'edgeotsu_pmedian5_1e4_573_neghalfsig_476_200',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 300
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian5_1e4_573_neghalfsigma_476_300,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian5_1e4_573_neghalfsig_476_300',
                                description = 'edgeotsu_pmedian5_1e4_573_neghalfsig_476_300',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 350
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian5_1e4_573_neghalfsig_476_350,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian5_1e4_573_neghalfsig_476_350',
                                description = 'edgeotsu_pmedian5_1e4_573_neghalfsig_476_350',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

##################### P-Median 5, thresh_no_data mu ########################

#ConnPix 50
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian5_1e4_573_mu_575_50,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian5_1e4_573_mu_575_50',
                                description = 'edgeotsu_pmedian5_1e4_573_mu_575_50',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 100
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian5_1e4_573_mu_575_100,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian5_1e4_573_mu_575_100',
                                description = 'edgeotsu_pmedian5_1e4_573_mu_575_100',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 200
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian5_1e4_573_mu_575_200,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian5_1e4_573_mu_575_200',
                                description = 'edgeotsu_pmedian5_1e4_573_mu_575_200',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 300
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian5_1e4_573_mu_575_300,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian5_1e4_573_mu_575_300',
                                description = 'edgeotsu_pmedian5_1e4_573_mu_575_300',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 350
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian5_1e4_573_mu_575_350,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian5_1e4_573_mu_575_350',
                                description = 'edgeotsu_pmedian5_1e4_573_mu_575_350',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

##################### P-Median 5, thresh_no_data mu + sigma/2 ########################


#ConnPix 50
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian5_1e4_573_poshalfsig_674_50,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian5_1e4_573_poshalfsig_674_50',
                                description = 'edgeotsu_pmedian5_1e4_573_poshalfsig_674_50',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 100
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian5_1e4_573_poshalfsig_674_100,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian5_1e4_573_poshalfsig_674_100',
                                description = 'edgeotsu_pmedian5_1e4_573_poshalfsig_674_100',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 200
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian5_1e4_573_poshalfsig_674_200,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian5_1e4_573_poshalfsig_674_200',
                                description = 'edgeotsu_pmedian5_1e4_573_poshalfsig_674_200',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 300
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian5_1e4_573_poshalfsig_674_300,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian5_1e4_573_poshalfsig_674_300',
                                description = 'edgeotsu_pmedian5_1e4_573_poshalfsig_674_300',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 350
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian5_1e4_573_poshalfsig_674_350,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian5_1e4_573_poshalfsig_674_350',
                                description = 'edgeotsu_pmedian5_1e4_573_poshalfsig_674_350',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

##################### P-Median 5, thresh_no_data mu + sigma ########################
#ConnPix 50
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian5_1e4_573_possig_773_50,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian5_1e4_573_possig_773_50',
                                description = 'edgeotsu_pmedian5_1e4_573_possig_773_50',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 100
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian5_1e4_573_possig_773_100,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian5_1e4_573_possig_773_100',
                                description = 'edgeotsu_pmedian5_1e4_573_possig_773_100',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 200
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian5_1e4_573_possig_773_200,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian5_1e4_573_possig_773_200',
                                description = 'edgeotsu_pmedian5_1e4_573_possig_773_200',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 300
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian5_1e4_573_possig_773_300,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian5_1e4_573_possig_773_300',
                                description = 'edgeotsu_pmedian5_1e4_573_possig_773_300',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 350
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian5_1e4_573_possig_773_350,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian5_1e4_573_possig_773_350',
                                description = 'edgeotsu_pmedian5_1e4_573_possig_773_350',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

### Step 5c: P-Median Window of 7

In [ ]:
##################### P-Median 7, thresh_no_data mu - sigma ########################


geemap.ee_export_image_to_asset(image = edgeotsu_pmedian7_1e4_568_negsig_387_50,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian7_1e4_568_negsig_387_50',
                                description = 'edgeotsu_pmedian7_1e4_568_negsig_387_50',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 100
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian7_1e4_568_negsig_387_100,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian7_1e4_568_negsig_387_100',
                                description = 'edgeotsu_pmedian7_1e4_568_negsig_387_100',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 200
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian7_1e4_568_negsig_387_200,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian7_1e4_568_negsig_387_200',
                                description = 'edgeotsu_pmedian7_1e4_568_negsig_387_200',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 300
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian7_1e4_568_negsig_387_300,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian7_1e4_568_negsig_387_300',
                                description = 'edgeotsu_pmedian7_1e4_568_negsig_387_300',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 350
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian7_1e4_568_negsig_387_350,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian7_1e4_568_negsig_387_350',
                                description = 'edgeotsu_pmedian7_1e4_568_negsig_387_350',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)


##################### P-Median 7, thresh_no_data mu - sigma/2 ########################

geemap.ee_export_image_to_asset(image = edgeotsu_pmedian7_1e4_568_neghalfsig_479_50,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian7_1e4_568_neghalfsig_479_50',
                                description = 'edgeotsu_pmedian7_1e4_568_neghalfsig_479_50',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 100
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian7_1e4_568_neghalfsig_479_100,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian7_1e4_568_neghalfsig_479_100',
                                description = 'edgeotsu_pmedian7_1e4_568_neghalfsig_479_100',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 200
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian7_1e4_568_neghalfsig_479_200,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian7_1e4_568_neghalfsig_479_200',
                                description = 'edgeotsu_pmedian7_1e4_568_neghalfsig_479_200',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 300
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian7_1e4_568_neghalfsig_479_300,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian7_1e4_568_neghalfsig_479_300',
                                description = 'edgeotsu_pmedian7_1e4_568_neghalfsig_479_300',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 350
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian7_1e4_568_neghalfsig_479_350,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian7_1e4_568_neghalfsig_479_350',
                                description = 'edgeotsu_pmedian7_1e4_568_neghalfsig_479_350',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)


##################### P-Median 7, thresh_no_data mu ########################
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian7_1e4_568_mu_571_50,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian7_1e4_568_mu_571_50',
                                description = 'edgeotsu_pmedian7_1e4_568_mu_571_50',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 100
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian7_1e4_568_mu_571_100,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian7_1e4_568_mu_571_100',
                                description = 'edgeotsu_pmedian7_1e4_568_mu_571_100',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 200
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian7_1e4_568_mu_571_200,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian7_1e4_568_mu_571_200',
                                description = 'edgeotsu_pmedian7_1e4_568_mu_571_200',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 300
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian7_1e4_568_mu_571_300,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian7_1e4_568_mu_571_300',
                                description = 'edgeotsu_pmedian7_1e4_568_mu_571_300',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 350
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian7_1e4_568_mu_571_350,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian7_1e4_568_mu_571_350',
                                description = 'edgeotsu_pmedian7_1e4_568_mu_571_350',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)


##################### P-Median 7, thresh_no_data mu + sigma/2 ########################

geemap.ee_export_image_to_asset(image = edgeotsu_pmedian7_1e4_568_poshalfsig_663_50,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian7_1e4_568_poshalfsig_663_50',
                                description = 'edgeotsu_pmedian7_1e4_568_poshalfsig_663_50',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 100
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian7_1e4_568_poshalfsig_663_100,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian7_1e4_568_poshalfsig_663_100',
                                description = 'edgeotsu_pmedian7_1e4_568_poshalfsig_663_100',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 200
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian7_1e4_568_poshalfsig_663_200,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian7_1e4_568_poshalfsig_663_200',
                                description = 'edgeotsu_pmedian7_1e4_568_poshalfsig_663_200',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 300
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian7_1e4_568_poshalfsig_663_300,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian7_1e4_568_poshalfsig_663_300',
                                description = 'edgeotsu_pmedian7_1e4_568_poshalfsig_663_300',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 350
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian7_1e4_568_poshalfsig_663_350,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian7_1e4_568_poshalfsig_663_350',
                                description = 'edgeotsu_pmedian7_1e4_568_poshalfsig_663_350',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

##################### P-Median 7, thresh_no_data mu + sigma ########################

geemap.ee_export_image_to_asset(image = edgeotsu_pmedian7_1e4_568_possig_754_50,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian7_1e4_568_possig_754_50',
                                description = 'edgeotsu_pmedian7_1e4_568_possig_754_50',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 100
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian7_1e4_568_possig_754_100,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian7_1e4_568_possig_754_100',
                                description = 'edgeotsu_pmedian7_1e4_568_possig_754_100',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 200
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian7_1e4_568_possig_754_200,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian7_1e4_568_possig_754_200',
                                description = 'edgeotsu_pmedian7_1e4_568_possig_754_200',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 300
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian7_1e4_568_possig_754_300,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian7_1e4_568_possig_754_300',
                                description = 'edgeotsu_pmedian7_1e4_568_possig_754_300',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 350
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian7_1e4_568_possig_754_350,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian7_1e4_568_possig_754_350',
                                description = 'edgeotsu_pmedian7_1e4_568_possig_754_350',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

### Step 5d: P-Median Window of 9

In [ ]:
##################### P-Median 9, thresh_no_data mu - sigma ########################


geemap.ee_export_image_to_asset(image = edgeotsu_pmedian9_1e4_566_negsig_394_50,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian9_1e4_566_negsig_394_50',
                                description = 'edgeotsu_pmedian9_1e4_566_negsig_394_50',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 100
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian9_1e4_566_negsig_394_100,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian9_1e4_566_negsig_394_100',
                                description = 'edgeotsu_pmedian9_1e4_566_negsig_394_100',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 200
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian9_1e4_566_negsig_394_200,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian9_1e4_566_negsig_394_200',
                                description = 'edgeotsu_pmedian9_1e4_566_negsig_394_200',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 300
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian9_1e4_566_negsig_394_300,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian9_1e4_566_negsig_394_300',
                                description = 'edgeotsu_pmedian9_1e4_566_negsig_394_300',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 350
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian9_1e4_566_negsig_394_350,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian9_1e4_566_negsig_394_350',
                                description = 'edgeotsu_pmedian9_1e4_566_negsig_394_350',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

##################### P-Median 9, thresh_no_data mu - sigma/2 ########################

geemap.ee_export_image_to_asset(image = edgeotsu_pmedian9_1e4_566_neghalfsig_481_50,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian9_1e4_566_neghalfsig_481_50',
                                description = 'edgeotsu_pmedian9_1e4_566_neghalfsig_481_50',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 100
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian9_1e4_566_neghalfsig_481_100,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian9_1e4_566_neghalfsig_481_100',
                                description = 'edgeotsu_pmedian9_1e4_566_neghalfsig_481_100',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 200
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian9_1e4_566_neghalfsig_481_200,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian9_1e4_566_neghalfsig_481_200',
                                description = 'edgeotsu_pmedian9_1e4_566_neghalfsig_481_200',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 300
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian9_1e4_566_neghalfsig_481_300,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian9_1e4_566_neghalfsig_481_300',
                                description = 'edgeotsu_pmedian9_1e4_566_neghalfsig_481_300',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 350
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian9_1e4_566_neghalfsig_481_350,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian9_1e4_566_neghalfsig_481_350',
                                description = 'edgeotsu_pmedian9_1e4_566_neghalfsig_481_350',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

##################### P-Median 9, thresh_no_data mu ########################


geemap.ee_export_image_to_asset(image = edgeotsu_pmedian9_1e4_566_mu_568_50,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian9_1e4_mu_568_50',
                                description = 'edgeotsu_pmedian9_1e4_566_mu_568_50',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 100
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian9_1e4_566_mu_568_100,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian9_1e4_mu_568_100',
                                description = 'edgeotsu_pmedian9_1e4_566_mu_568_100',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 200
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian9_1e4_566_mu_568_200,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian9_1e4_mu_568_200',
                                description = 'edgeotsu_pmedian9_1e4_566_mu_568_200',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 300
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian9_1e4_566_mu_568_300,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian9_1e4_566_mu_568_300',
                                description = 'edgeotsu_pmedian9_1e4_566_mu_568_300',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 350
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian9_1e4_566_mu_568_350,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian9_1e4_566_mu_568_350',
                                description = 'edgeotsu_pmedian9_1e4_566_mu_568_350',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

##################### P-Median 9, thresh_no_data mu + sigma/2 ########################

geemap.ee_export_image_to_asset(image = edgeotsu_pmedian9_1e4_566_poshalfsig_655_50,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian9_1e4_566_poshalfsig_655_50',
                                description = 'edgeotsu_pmedian9_1e4_566_poshalfsig_655_50',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 100
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian9_1e4_566_poshalfsig_655_100,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian9_1e4_566_poshalfsig_655_100',
                                description = 'edgeotsu_pmedian9_1e4_566_poshalfsig_655_100',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 200
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian9_1e4_566_poshalfsig_655_200,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian9_1e4_566_poshalfsig_655_200',
                                description = 'edgeotsu_pmedian9_1e4_566_poshalfsig_655_200',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 300
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian9_1e4_566_poshalfsig_655_300,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian9_1e4_566_poshalfsig_655_300',
                                description = 'edgeotsu_pmedian9_1e4_566_poshalfsig_655_300',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 350
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian9_1e4_566_poshalfsig_655_350,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian9_1e4_566_poshalfsig_655_350',
                                description = 'edgeotsu_pmedian9_1e4_566_poshalfsig_655_350',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

##################### P-Median 9, thresh_no_data mu + sigma ########################
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian9_1e4_566_possig_742_50,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian9_1e4_566_possig_742_50',
                                description = 'edgeotsu_pmedian9_1e4_566_possig_742_50',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 100
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian9_1e4_566_possig_742_100,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian9_1e4_566_possig_742_100',
                                description = 'edgeotsu_pmedian9_1e4_566_possig_742_100',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 200
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian9_1e4_566_possig_742_200,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian9_1e4_566_possig_742_200',
                                description = 'edgeotsu_pmedian9_1e4_566_possig_742_200',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 300
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian9_1e4_566_possig_742_300,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian9_1e4_566_possig_742_300',
                                description = 'edgeotsu_pmedian9_1e4_566_possig_742_300',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 350
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian9_1e4_566_possig_742_350,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian9_1e4_566_possig_742_350',
                                description = 'edgeotsu_pmedian9_1e4_566_possig_742_350',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

### Step 5e: P-Median Window of 11

In [ ]:
##################### P-Median 11, thresh_no_data mu - sigma ########################
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian11_1e4_563_negsig_394_50,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian11_1e4_563_negsig_394_50',
                                description = 'edgeotsu_pmedian11_1e4_563_negsig_394_50',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 100
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian11_1e4_563_negsig_394_100,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian11_1e4_563_negsig_394_100',
                                description = 'edgeotsu_pmedian11_1e4_563_negsig_394_100',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 200
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian11_1e4_563_negsig_394_200,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian11_1e4_563_negsig_394_200',
                                description = 'edgeotsu_pmedian11_1e4_563_negsig_394_200',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

In [ ]:
##################### P-Median 11, thresh_no_data mu - sigma ########################
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian11_1e4_563_negsig_394_50,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian11_1e4_563_negsig_394_50',
                                description = 'edgeotsu_pmedian11_1e4_563_negsig_394_50',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 100
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian11_1e4_563_negsig_394_100,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian11_1e4_563_negsig_394_100',
                                description = 'edgeotsu_pmedian11_1e4_563_negsig_394_100',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 200
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian11_1e4_563_negsig_394_200,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian11_1e4_563_negsig_394_200',
                                description = 'edgeotsu_pmedian11_1e4_563_negsig_394_200',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 300
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian11_1e4_563_negsig_394_300,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian11_1e4_563_negsig_394_300',
                                description = 'edgeotsu_pmedian11_1e4_563_negsig_394_300',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 350
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian11_1e4_563_negsig_394_350,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian11_1e4_563_negsig_394_350',
                                description = 'edgeotsu_pmedian11_1e4_563_negsig_394_350',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)
##################### P-Median 11, thresh_no_data mu - sigma/2 ########################

geemap.ee_export_image_to_asset(image = edgeotsu_pmedian11_1e4_563_neghalfsig_482_50,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian11_1e4_563_neghalfsig_482_50',
                                description = 'edgeotsu_pmedian11_1e4_563_neghalfsig_482_50',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 100
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian11_1e4_563_neghalfsig_482_100,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian11_1e4_563_neghalfsig_482_100',
                                description = 'edgeotsu_pmedian11_1e4_563_neghalfsig_482_100',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 200
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian11_1e4_563_neghalfsig_482_200,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian11_1e4_563_neghalfsig_482_200',
                                description = 'edgeotsu_pmedian11_1e4_563_neghalfsig_482_200',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 300
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian11_1e4_563_neghalfsig_482_300,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian11_1e4_563_neghalfsig_482_300',
                                description = 'edgeotsu_pmedian11_1e4_563_neghalfsig_482_300',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 350
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian11_1e4_563_neghalfsig_482_350,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian11_1e4_563_neghalfsig_482_350',
                                description = 'edgeotsu_pmedian11_1e4_563_neghalfsig_482_350',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

##################### P-Median 11, thresh_no_data mu ########################

geemap.ee_export_image_to_asset(image = edgeotsu_pmedian11_1e4_563_mu_566_50,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian11_1e4_563_mu_566_50',
                                description = 'edgeotsu_pmedian11_1e4_563_mu_566_50',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 100
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian11_1e4_563_mu_566_100,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian11_1e4_563_mu_566_100',
                                description = 'edgeotsu_pmedian11_1e4_563_mu_566_100',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 200
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian11_1e4_563_mu_566_200,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian11_1e4_563_mu_566_200',
                                description = 'edgeotsu_pmedian11_1e4_563_mu_566_200',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 300
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian11_1e4_563_mu_566_300,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian11_1e4_563_mu_566_300',
                                description = 'edgeotsu_pmedian11_1e4_563_mu_566_300',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 350
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian11_1e4_563_mu_566_350,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian11_1e4_563_mu_566_350',
                                description = 'edgeotsu_pmedian11_1e4_563_mu_566_350',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

##################### P-Median 11, thresh_no_data mu + sigma/2 ########################

geemap.ee_export_image_to_asset(image = edgeotsu_pmedian11_1e4_563_poshalfsig_649_50,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian11_1e4_563_poshalfsig_649_50',
                                description = 'edgeotsu_pmedian11_1e4_563_poshalfsig_649_50',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 100
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian11_1e4_563_poshalfsig_649_100,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian11_1e4_563_poshalfsig_649_100',
                                description = 'edgeotsu_pmedian11_1e4_563_poshalfsig_649_100',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 200
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian11_1e4_563_poshalfsig_649_200,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian11_1e4_563_poshalfsig_649_200',
                                description = 'edgeotsu_pmedian11_1e4_563_poshalfsig_649_200',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 300
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian11_1e4_563_poshalfsig_649_300,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian11_1e4_563_poshalfsig_649_300',
                                description = 'edgeotsu_pmedian11_1e4_563_poshalfsig_649_300',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 350
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian11_1e4_563_poshalfsig_649_350,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian11_1e4_563_poshalfsig_649_350',
                                description = 'edgeotsu_pmedian11_1e4_563_poshalfsig_649_350',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

##################### P-Median 11, thresh_no_data mu + sigma ########################
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian11_1e4_563_mu_566_50,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian11_1e4_563_mu_566_50',
                                description = 'edgeotsu_pmedian11_1e4_563_mu_566_50',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 100
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian11_1e4_563_mu_566_100,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian11_1e4_563_mu_566_100',
                                description = 'edgeotsu_pmedian11_1e4_563_mu_566_100',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 200
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian11_1e4_563_mu_566_200,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian11_1e4_563_mu_566_200',
                                description = 'edgeotsu_pmedian11_1e4_563_mu_566_200',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 300
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian11_1e4_563_mu_566_300,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian11_1e4_563_mu_566_300',
                                description = 'edgeotsu_pmedian11_1e4_563_mu_566_300',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 350
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian11_1e4_563_mu_566_350,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian11_1e4_563_mu_566_350',
                                description = 'edgeotsu_pmedian11_1e4_563_mu_566_350',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

##################### P-Median 11, thresh_no_data mu + sigma/2 ########################

geemap.ee_export_image_to_asset(image = edgeotsu_pmedian11_1e4_563_possig_732_50,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian11_1e4_563_possig_732_50',
                                description = 'edgeotsu_pmedian11_1e4_563_possig_732_50',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 100
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian11_1e4_563_possig_732_100,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian11_1e4_563_possig_732_100',
                                description = 'edgeotsu_pmedian11_1e4_563_possig_732_100',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 200
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian11_1e4_563_possig_732_200,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian11_1e4_563_possig_732_200',
                                description = 'edgeotsu_pmedian11_1e4_563_possig_732_200',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 300
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian11_1e4_563_possig_732_300,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian11_1e4_563_possig_732_300',
                                description = 'edgeotsu_pmedian11_1e4_563_possig_732_300',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 350
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian11_1e4_563_possig_732_350,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian11_1e4_563_possig_732_350',
                                description = 'edgeotsu_pmedian11_1e4_563_possig_732_350',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

In [ ]:

#ConnPix 350
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian11_1e4_563_mu_566_350,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian11_1e4_563_mu_566_350',
                                description = 'edgeotsu_pmedian11_1e4_563_mu_566_350',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

##################### P-Median 11, thresh_no_data mu + sigma/2 ########################

geemap.ee_export_image_to_asset(image = edgeotsu_pmedian11_1e4_563_poshalfsig_649_50,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian11_1e4_563_poshalfsig_649_50',
                                description = 'edgeotsu_pmedian11_1e4_563_poshalfsig_649_50',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 100
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian11_1e4_563_poshalfsig_649_100,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian11_1e4_563_poshalfsig_649_100',
                                description = 'edgeotsu_pmedian11_1e4_563_poshalfsig_649_100',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 200
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian11_1e4_563_poshalfsig_649_200,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian11_1e4_563_poshalfsig_649_200',
                                description = 'edgeotsu_pmedian11_1e4_563_poshalfsig_649_200',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 300
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian11_1e4_563_poshalfsig_649_300,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian11_1e4_563_poshalfsig_649_300',
                                description = 'edgeotsu_pmedian11_1e4_563_poshalfsig_649_300',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 350
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian11_1e4_563_poshalfsig_649_350,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian11_1e4_563_poshalfsig_649_350',
                                description = 'edgeotsu_pmedian11_1e4_563_poshalfsig_649_350',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

##################### P-Median 11, thresh_no_data mu + sigma ########################
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian11_1e4_563_mu_566_50,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian11_1e4_563_mu_566_50',
                                description = 'edgeotsu_pmedian11_1e4_563_mu_566_50',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 100
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian11_1e4_563_mu_566_100,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian11_1e4_563_mu_566_100',
                                description = 'edgeotsu_pmedian11_1e4_563_mu_566_100',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 200
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian11_1e4_563_mu_566_200,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian11_1e4_563_mu_566_200',
                                description = 'edgeotsu_pmedian11_1e4_563_mu_566_200',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 300
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian11_1e4_563_mu_566_300,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian11_1e4_563_mu_566_300',
                                description = 'edgeotsu_pmedian11_1e4_563_mu_566_300',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 350
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian11_1e4_563_mu_566_350,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian11_1e4_563_mu_566_350',
                                description = 'edgeotsu_pmedian11_1e4_563_mu_566_350',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

##################### P-Median 11, thresh_no_data mu + sigma/2 ########################

geemap.ee_export_image_to_asset(image = edgeotsu_pmedian11_1e4_563_possig_732_50,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian11_1e4_563_possig_732_50',
                                description = 'edgeotsu_pmedian11_1e4_563_possig_732_50',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 100
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian11_1e4_563_possig_732_100,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian11_1e4_563_possig_732_100',
                                description = 'edgeotsu_pmedian11_1e4_563_possig_732_100',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 200
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian11_1e4_563_possig_732_200,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian11_1e4_563_possig_732_200',
                                description = 'edgeotsu_pmedian11_1e4_563_possig_732_200',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 300
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian11_1e4_563_possig_732_300,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian11_1e4_563_possig_732_300',
                                description = 'edgeotsu_pmedian11_1e4_563_possig_732_300',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

#ConnPix 350
geemap.ee_export_image_to_asset(image = edgeotsu_pmedian11_1e4_563_possig_732_350,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/perturbations/edgeotsu_pmedian11_1e4_563_possig_732_350',
                                description = 'edgeotsu_pmedian11_1e4_563_possig_732_350',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

In [ ]:
# Load your SAR image from assets
#asset_id1 = "users/mickymags/watchduty/alaska_iceye_v2/kipnuk_1014_0112"

#image = ee.Image(asset_id1)
#image = image.rename('VV', 'angle')

# Get band names
#bands = image.select('VV').bandNames().getInfo()
#print("Bands in image:", bands)

In [ ]:
'''
# Create a histogram for each band
def plot_histograms(image, bands,  aoi_descriptor, region, xmin, xmax, scale=3.46325, n_bins=4096):
    """Plot histograms for each band of an ee.Image."""
    fig, axes = plt.subplots(len(bands), 1, figsize=(8, 4*len(bands)))

    if len(bands) == 1:
        axes = [axes]

    for i, band in enumerate(bands):
        # Reduce region to histogram
        hist_dict = image.select(band).reduceRegion(
            reducer=ee.Reducer.histogram(maxBuckets=n_bins), #maxBuckets=n_bins),
            geometry=region,
            scale=scale,

            bestEffort=True
        ).get(band).getInfo()

        if hist_dict is not None:
            counts = hist_dict['histogram']
            bucketMeans = hist_dict['bucketMeans']

            axes[i].bar(bucketMeans, counts, width=(bucketMeans[1]-bucketMeans[0]))
            axes[i].set_title(f"Histogram of {band} for {aoi_descriptor}")
            axes[i].set_xlabel("Pixel values")
            axes[i].set_ylabel("Frequency")
            axes[i].set_xlim(xmin, xmax) #0, 250
        else:
            axes[i].set_title(f"{band}: No data")

    plt.grid()
    plt.tight_layout()
    plt.show()
'''

In [ ]:
sent1 = ee.ImageCollection("COPERNICUS/S1_GRD").filterBounds(aoi_geom).filterDate("2025-10-22", "2025-10-23").select(["VV", "angle"]).first().clip(aoi_geom)
s1_rl = hf.refined_lee(sent1)

In [ ]:
plot_histograms(s1_rl, bands, 'Sentinel-1 Image', aoi_geom, -30, 0, scale = 30, n_bins = 256)

In [ ]:
plot_histograms(image, bands, 'Raw ICEYE Image 1', aoi_geom, 0, 3000, scale = 4)

In [ ]:
plot_histograms(ic_lee_med, bands, 'ICEYE Image 1 (10/12)', aoi_geom, 0, 3000, scale = 4, n_bins = 256)

In [ ]:
plot_histograms(ic_refined_med, bands, 'Refined-Lee Image', aoi_geom, 250, 400, scale = 4, n_bins = 48)

In [ ]:
plot_histograms(ic_gamma_med, bands, 'Gamma-Map Filtered Image', aoi_geom, 0, 3000, scale = 4, n_bins = 48)

In [ ]:
plot_histograms(ic_pmedian_med, bands, 'ICEYE PMedian', aoi_geom, 0, 3000, scale = 4, n_bins = 2048)

In [ ]:
Map = geemap.Map(center = (lat, lon), zoom = 11)
Map.addLayer(aoi_geom, {}, 'Area of Interest')
Map.addLayer(iceye_med, iceye_vp, 'ICEYE2')
Map.addLayer(ic_pmedian_med, iceye_vp, 'ICEYE2 P Median')
Map.addLayer(ic_lee_med, iceye_vp, 'ICEYE Lee')
Map.addLayer(ic_gamma_med, iceye_vp, 'ICEYE Gamma')
#Map.addLayer(s1_rl, iceye_vp, 'Sentinel-1 Refined Lee')
#Map.addLayer(iceye2_lee, iceye_vp, 'ICEYE2 Lee')
#Map.addLayer(iceye2_gamma, iceye_vp, 'ICEYE2 Gamma')
#Map.addLayer(iceye2_refined, iceye_vp, 'ICEYE2 Refined')

Map.addLayerControl()
Map

# Part 3: Edge Otsu Algorithm Perturbations

Now that we have several different initial thresholds for each different speckle window size, we need to tune the other parameters in the edge otsu algorithm. These are the connected_pixels

### Raw Image

In [ ]:
import numpy as np

In [ ]:
iceye_sampled = iceye_med.sample(aoi, numPixels = 5000, scale = 4).getInfo()
features = iceye_sampled['features']
first_feature = features[0]
first_feature['properties']['VV']

vals = []

for j in range(len(features)):
  feat_of_int = features[j]
  vals.append(feat_of_int['properties']['VV'])

np_vals = np.array(vals)
np_vals.mean()

In [ ]:
edge = hf.edge_otsu(
    iceye_med, #iceye_med
    band = 'VV',
    region = aoi,
    edge_buffer = 30,
    initial_threshold = np_vals.mean(),
    thresh_no_data = np_vals.mean()+50,
    scale = 4
)

### P-Median Image

In [ ]:
iceye_sampled = ic_pmedian_med.sample(aoi, numPixels = 5000, scale = 4).getInfo()
features = iceye_sampled['features']
first_feature = features[0]
first_feature['properties']['VV']

pmed_vals = []

for j in range(len(features)):
  feat_of_int = features[j]
  pmed_vals.append(feat_of_int['properties']['VV'])

np_pmed = np.array(pmed_vals)
np_pmed.mean()

In [ ]:
edge_median = hf.edge_otsu(
    ic_pmedian_med, #iceye_med
    band = 'VV',
    region = aoi,
    edge_buffer = 30,
    initial_threshold = np_pmed.mean(),
    thresh_no_data = np_pmed.mean()+50,
)

# Lee-Sigma Image

In [ ]:
iceye_sampled = ic_lee_med.sample(aoi, numPixels = 5000, scale = 4).getInfo()
features = iceye_sampled['features']
first_feature = features[0]
first_feature['properties']['VV']

lee_vals = []

for j in range(len(features)):
  feat_of_int = features[j]
  lee_vals.append(feat_of_int['properties']['VV'])

np_lee = np.array(lee_vals)
np_lee.mean()

In [ ]:
edge_lee = hf.edge_otsu(
    ic_lee_med, #iceye_med
    band = 'VV',
    region = aoi,
    edge_buffer = 30,
    initial_threshold = np_lee.mean(),
    thresh_no_data = np_lee.mean()+50,
)

# Refined-Lee Image

In [ ]:
iceye_refined_sampled = ic_refined_med.sample(aoi, numPixels = 5000, scale = 4).getInfo()
refined_features = iceye_refined_sampled['features']
first_re_feature = refined_features[0]
first_re_feature['properties']['VV']

refined_vals = []

for j in range(len(refined_features)):
  refined_feat_of_int = refined_features[j]
  refined_vals.append(refined_feat_of_int['properties']['VV'])

np_refined = np.array(refined_vals)
np_refined.mean()

In [ ]:
edge_refined = hf.edge_otsu(
    ic_refined_med, #iceye_med
    band = 'VV',
    region = aoi,
    edge_buffer = 30,
    initial_threshold = np_refined.mean(),
    thresh_no_data = np_refined.mean()+50,
)

### Gamma MAP Image

In [ ]:
iceye_gamma_sampled = ic_gamma_med.sample(aoi, numPixels = 5000, scale = 4).getInfo()
gamma_features = iceye_gamma_sampled['features']
first_gamma_feature = refined_features[0]
first_gamma_feature['properties']['VV']

gamma_vals = []

for j in range(len(gamma_features)):
  gamma_feat_of_int = gamma_features[j]
  gamma_vals.append(gamma_feat_of_int['properties']['VV'])

np_gamma = np.array(gamma_vals)
np_gamma.mean()

In [ ]:
edge_gamma = hf.edge_otsu(
    ic_gamma_med, #iceye_med
    band = 'VV',
    region = aoi,
    edge_buffer = 30,
    initial_threshold = np_gamma.mean(),
    thresh_no_data = np_gamma.mean()+50,
)

### Map Visualization

In [ ]:
water_vp = {
    'palette': ['000000', 'add8e6'],
    'min': 0,
    'max': 1
}

In [ ]:
Map = geemap.Map(center = (lat, lon), zoom = 12)
Map.addLayer(aoi, {}, 'Area of Interest')
Map.addLayer(iceye_med, iceye_vp, 'ICEYE')
Map.addLayer(edge, water_vp, 'Edge')
Map.addLayer(edge_median, water_vp, 'Edge Median')
Map.addLayer(edge_lee, water_vp, 'Edge Lee')
Map.addLayer(edge_refined, water_vp, 'Edge Refined')
Map.addLayerControl()
Map

# Exporting Images to GEE Assets

In [ ]:
geemap.ee_export_image_to_asset(image =edge,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/edge_no_tc_no_sf',
                                description = 'iceye_edge_otsu_no_speckle_filter',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

In [ ]:
geemap.ee_export_image_to_asset(image = edge_lee,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/edge_no_tc_leesigma_sf',
                                description = 'edge_no_tc_leesigma_sf',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

In [ ]:
geemap.ee_export_image_to_asset(image = edge_median,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/edge_no_tc_median_sf',
                                description = 'iedge_no_tc_median_sf',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

In [ ]:
geemap.ee_export_image_to_asset(image = edge_refined,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/edge_no_tc_refined_sf',
                                description = 'edge_no_tc_refined_sf',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

In [ ]:
geemap.ee_export_image_to_asset(image = edge_gamma,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/edge_no_tc_gamma_sf',
                                description = 'edge_no_tc_gamma_sf',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

# Exporting Images to Google Drive

In [ ]:
geemap.ee_export_image_to_drive(image=edge_gamma,
                                folder='WatchDuty',
                                description='gdrive_edge_no_tc_gamma_sf',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels=1e13)

In [ ]:
geemap.ee_export_image_to_drive(image=edge,
                                folder='WatchDuty',
                                description='gdrive_edge_no_tc_no_sf',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels=1e13)

In [ ]:
geemap.ee_export_image_to_drive(image=edge_lee,
                                folder='WatchDuty',
                                description='gdrive_edge_no_tc_leesigma_sf',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels=1e13)

In [ ]:
geemap.ee_export_image_to_drive(image=edge_refined,
                                folder='WatchDuty',
                                description='gdrive_edge_no_tc_refined_sf',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels=1e13)

In [ ]:
geemap.ee_export_image_to_drive(image=edge_median,
                                folder='WatchDuty',
                                description='gdrive_edge_no_tc_median_sf',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels=1e13)